# CIFAR-100 knowledge distillation study

Run All in a fresh kernel. Qualification exercises six student configurations, a deliberate crash and a fresh-process restart on disposable data. Measured mode runs the scientific teacher and the chosen seed blocks. Final review opens the test labels only after all 30 primary student runs and their evidence audits exist.

In [1]:
MODE = "qualification"  # qualification, measured, final_review
HF_DATASET_REPO = ""  # Existing dedicated owner/repo, enabled for the Kaggle HF_TOKEN secret
CIFAR100_ARCHIVE = None  # Auto-detect one original cifar-100-binary.tar.gz in /kaggle/input
SEED_BLOCKS = (42,)  # All primary blocks: (42, 123, 2026, 31415, 27182)

## Embedded, verified runtime

The notebook carries all experiment sources and the protocol. No source repository or model download is required.

In [2]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
from tempfile import gettempdir
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
SOURCES = {"PROTOCOL.md": "# Experimental protocol\n\nVersion 1.0 \u00b7 22 November 2026 \u00b7 specified before benchmark training. Freeze a digest of this document, the implementation and the input manifests before the first measured student update. Later changes receive a dated amendment.\n\n## 1. Primary comparison\n\nFor one fixed ResNet-34 teacher, estimate the change in final ResNet-18 accuracy when extra teacher views go to source images with the largest two-view Jensen\u2013Shannon divergence. The comparator randomly assigns exactly the same six/two quotas within each fine class. The estimand includes the resulting differences in the training views. A claim that soft labels alone caused a gain would require additional hard-label runs on the identical finite view banks.\n\nIllustration on CIFAR-100: each fine class contributes 450 fitting images. Uniform allocation buys 450 \u00d7 4 = 1,800 teacher evaluations per class. Unequal allocation buys 225 \u00d7 6 + 225 \u00d7 2 = 1,800. Across 100 fine classes, the total is 180,000. One query means one augmented image forwarded through the teacher, irrespective of batching.\n\n## 2. Datasets and splits\n\n| Dataset | Native input | Fitting images | Validation candidates | Final cohort before exact-overlap audit |\n| --- | --- | ---: | ---: | --- |\n| CIFAR-100 | 32 \u00d7 32 RGB, 100 fine classes | 45,000, 450/class | 5,000 | 10,000 official test images |\n| CIFAR-10 | 32 \u00d7 32 RGB, 10 classes | 45,000, 4,500/class | 5,000 | 10,000 official test images |\n| Tiny ImageNet-200 | 64 \u00d7 64 RGB, 200 synsets | 90,000, 450/class | 10,000 from official training | 10,000 labeled official validation images, reserved as the study's final holdout |\n\nWithin each fine class, sort official training IDs by SHA-256 of the UTF-8 string `kd-view-v1|1729|<image_id>`, tie-breaking on the canonical ID. Use the first fitting quota and reserve the remainder for validation. CIFAR-100 binary IDs are `train/00000` through `train/49999` in source order. CIFAR-10 uses official batch order followed by record order and the same five-digit ID convention. Tiny ImageNet uses the synset and original relative POSIX path, with class order fixed by wnids.txt. Use val_annotations.txt for its final labels. The challenge test directory has no role.\n\nHash decoded RGB uint8 in CHW order and record the native dimensions. Keep fitting exposure fixed. Exclude validation candidates matching fitting pixels and repeated validation pixels after the first canonical source ID. Exclude final images matching any official training candidate and repeated final pixels after the first. Freeze the resulting IDs without reading final predictions. Keep and report exclusion reasons, per-class eligible counts and standard unfiltered final-split results. Exact hashing does not identify every near-duplicate.\n\nCompute channel means and population standard deviations on unaugmented fitting pixels alone. For validation and final evaluation, use native resolution, RGB conversion and this normalization. Reserve final labels and metrics for one release after all predeclared reconfirmation blocks are fixed. Validation can be used for teacher selection and debugging. Version changes after seeing benchmark outcomes must disclose the development history.\n\n## 3. Networks and common training recipe\n\n| Role | Configuration | Purpose |\n| --- | --- | --- |\n| Teacher | torchvision ResNet-34, one per dataset, trained from scratch, seed 907 | One strong, fully identified target for all policies on that dataset |\n| Primary student | torchvision ResNet-18, trained from scratch | Same-family capacity reduction |\n| Cross-family student | timm MobileNetV4-Conv-Small with pretrained weights disabled | Tests whether allocation transfers to a different CNN on the same CIFAR-100 split |\n\nFor the two ResNets, use a 3 \u00d7 3 first convolution with 64 outputs, stride 1, padding 1, no bias, and an identity in place of the initial max pool. Keep all subsequent residual stages and global pooling. For MobileNetV4 keep the native convolutional stages and adaptive pooling, set the first convolution stride to 1, use the dataset class count, and set dropout and stochastic depth to zero. Record the exact variant, package version, input/output shapes, parameter counts and effective stage strides at qualification. The native 224-pixel weights and published device timings are not part of this comparison.\n\nTrain in FP32 with SGD, momentum 0.9, no Nesterov, weight decay 0.0005 on trainable tensors, global batch 128 on one active GPU. Include the final short batch. Use a peak learning rate of 0.1, a five-epoch linear warm-up and cosine decay to zero at the last update. For total updates U, warm-up W and zero-based update u, use 0.1(u+1)/W for u < W and 0.05[1+cos(\u03c0(u\u2212W)/(U\u2212W\u22121))] after warm-up. No gradient accumulation, mixed precision, label smoothing, MixUp, CutMix or per-policy tuning. Each seed shares identical starting weights and minibatch order across its arms. Disable nondeterministic kernels and record the actual software/hardware identity.\n\nTeachers use fresh crops, hard-label cross-entropy and 90 epochs. Pick the highest validation top-1, breaking ties by lower validation NLL and then earlier epoch. Record the validation margin over the supervised student once available. Retain weak or anomalous teachers in the audit. Students use 60 epochs, so every two-, four- and six-view bank can be traversed equally often. The epoch-60 checkpoint defines the main comparison. Save the validation-best checkpoint separately under its own identity. A short training horizon constrains the interpretation.\n\n## 4. Query allocation and loss\n\n1. Reflect-pad CIFAR by four pixels or Tiny ImageNet by eight. Enumerate all native-size integer crops and both horizontal flip states. Order the possible transforms per source image with NumPy PCG64 seeded from a stable hash of the block seed and source ID. Slots 0\u20135 are distinct transform keys. Every policy in a block shares their definition. Repeated pixels can still occur in uniform regions.\n2. Evaluate slots 0 and 1 with the frozen teacher in evaluation mode. Store dense raw FP32 logits. Calculate the two softmax vectors at scoring temperature 1. Let their mean be m. The disagreement score is half of KL(p\u2080\u2016m) + KL(p\u2081\u2016m), using natural log. The entropy control uses the mean of H(p\u2080) and H(p\u2081). Log teacher correctness on both fitting probes as a diagnostic.\n3. Rank within each ground-truth class. In a ranked arm, assign six views to the top half and two to the bottom half. In the shuffled arm, rank by a seeded hash independent of teacher outputs. In an odd-sized class the median gets four. Break score ties with an independent seeded hash and canonical ID. Evaluate only the extra slots assigned. Include both initial probes in each policy's budget. Never evaluate six views everywhere and then charge only the retained four.\n4. Replay the saved crop and flip that produced the target. Each fitting image appears once per epoch, in a block-specific shuffle shared across arms. Select slot (epoch + per-image offset) modulo its assigned count. Across 60 epochs a two-slot source has 30 exposures per slot, a four-slot source 15, and a six-slot source 10. Batching the flat cache rows uniformly would overweight six-slot sources and change the question.\n5. For the fresh arms, use the same per-epoch source order and draw one crop/flip independently from the same transform family using a separate keyed stream. The hard-label and live-KD arms share these draws. For KD, minimize (1\u2212\u03b1)CE(y,z\u209b) + \u03b1T\u00b2 KL(softmax(z\u209c/T) \u2016 softmax(z\u209b/T)), with \u03b1=0.5, T=4 and a batch-mean KL. The student and teacher see identical pixels. Cached logits are raw and can reproduce this loss. These values are fixed design choices.\n\n| Arm | Target production | Student exposure | Question answered |\n| --- | --- | --- | --- |\n| Hard labels | No teacher | Fresh views | How strong is supervised training? |\n| Live KD | Frozen teacher sees every fresh student view | Fresh views | What difference accompanies replacing the fresh view stream with a finite bank? |\n| Uniform four | Four cached aligned targets per image | Finite bank | Does choosing quotas beat a simple reusable cache? |\n| Shuffled six/two | Class-balanced, randomly assigned quotas | Finite bank | Does the identity of the images given extra views matter? |\n| Entropy six/two | Highest mean teacher entropy gets six | Finite bank | Is ordinary uncertainty sufficient? |\n| Disagreement six/two | Highest two-view disagreement gets six | Finite bank | Proposed allocation |\n\nLive KD and uniform caching differ in view diversity and target delivery. A no-fit replay test comparing both paths on exactly the same tensors handles the numerical equivalence question separately. The entropy arm and any supervised-versus-live comparison are secondary.\n\n## 5. Run sequence and realistic scale\n\n| Order | Dataset and student | Arms | Paired blocks | Student fits |\n| ---: | --- | --- | --- | ---: |\n| 1 | CIFAR-100 / ResNet-18 | All six above | 42, 123, 2026, 31415, 27182 | 30 |\n| 2 | CIFAR-100 / MobileNetV4-Conv-Small | Uniform, shuffled, disagreement | 42, 123, 2026 | 9 |\n| 3 | CIFAR-10 / ResNet-18 | Uniform, shuffled, disagreement | 42, 123, 2026 | 9 |\n| 4 | Tiny ImageNet / ResNet-18 | Uniform, shuffled, disagreement | 42, 123, 2026 | 9 |\n\nOne teacher per dataset adds three fits to the maximum of 57 students. Begin with the CIFAR-100 teacher and seed-42 arms. Count the seed-42 fits once if the protocol survives qualification. Qualify code on disposable fixtures first. Keep benchmark test labels sealed during the first block. Funding, time and engineering integrity can determine whether the later blocks execute. The sign of the pilot gain is not a reason to discard a seed or silently change the confirmatory set.\n\nThe [machine-readable run matrix](RUN_MATRIX.csv) fixes the order, cache ownership and planned query counts. Within each block it sorts arms by the SHA-256 of `order-v1|dataset|model|seed|arm` using the CSV spellings. CIFAR-100 cross-family runs reuse the corresponding same-seed banks. The complete matrix contains 38 distinct banks and 8,460,000 planned physical construction queries. The five full live-KD fits add 13,500,000 teacher training-view queries. These are budget calculations. The separate diagnostic forwards, parity checks, teacher validation, retries and actual energy must enter the observed ledger.\n\nAt batch 128, CIFAR has 352 updates per epoch and Tiny ImageNet 704. Each student therefore needs 21,120 or 42,240 updates, respectively. The maximum design totals 1,393,920 student updates and 126,720 teacher updates. Book time using the median seconds per update from a representative T4 dry run, plus separately timed cache construction, evaluation, recovery and publication. Do not substitute the local fixture speed. If resources permit only the first block, report it as a one-seed pilot.\n\n## 6. Statistical decisions and interpretation\n\nFor block s, compute \u0394\u209b = 100(correct_guided \u2212 correct_shuffled)/N on the identical final IDs. Always show the five values, raw correct counts and N. Report their mean, sample SD and two-sided 95% interval using t with four degrees of freedom. The interval describes paired training-block variation conditional on this teacher and finite evaluation cohort. Its interpretation is approximate with five blocks. Images, epochs and crops are not extra independent training replicates. Report per-image paired disagreements as a diagnostic.\n\nFirst test disagreement versus shuffled. Only if its lower bound exceeds zero, test disagreement versus uniform with the same interval in fixed order. A useful allocation recommendation requires both lower bounds above zero and a primary point estimate at least +0.30 percentage points. This design threshold corresponds to about 30 decisions per 10,000 images. If an upper bound falls below +0.30, that comparison excludes a gain of the specified practical magnitude. An interval spanning both zero and +0.30 is inconclusive. Report estimates and intervals even when formal testing stops. Show entropy, class and transfer comparisons descriptively. Multiple datasets sharing raw source families do not constitute independent populations.\n\nReport macro recall, per-class support/confusion, NLL, and top-label ECE with 15 equal-width confidence bins on [0,1], including confidence 1 in the last bin. Inspect high-score fitting images with correct and incorrect teacher probes. For a diagnostic independent of the ranking views, sample ten fitting IDs per primary CIFAR-100 class by a separate hash. On those 1,000 images measure disagreement on two new transform keys outside the six-slot bank. Report the within-class rank association and charge the 2,000 additional queries to diagnostics. This check cannot influence allocations or epochs.\n\n## 7. Actual energy and projected reuse\n\nFor each standalone strategy log physical teacher queries, unique cache rows, dense payload bytes, CSV/index and total disk bytes, wall time, GPU memory peaks and GPU-board joules for each phase. For the clean comparison, build each bank independently (including the first two queries for each image) and reuse it only for a declared second student on the same dataset/teacher/seed/view contract. Reuse between student architectures on CIFAR-100 is feasible by design. Charge the physical creation once in the actual research ledger. Show the full standalone preparation cost for each strategy's scenario. Include fixtures, failures and exploratory costs in the campaign ledger under separate phases.\n\nFor a fixed teacher and one reusable bank, let B be the measured active-board cache-build energy including ranking and serialization. Let L and C be mean measured live and cached per-fit energies, including reads, the same evaluation cadence and matched local evidence writing. A reused bank saves energy after r fits when B + rC < rL. If L > C, the first strict saving is max(1, floor(B/(L\u2212C))+1). If L \u2264 C and B \u2265 0 there is no finite saving. Give the observed single-fit total and show scenarios r = 1, 3, 5. For unequal students use the sum of their paired per-fit differences. Label extrapolations, report per-block values, and flag denominators close to zero.\n\nDescribe an energy saving as quality-preserving only when the lower paired 95% bound for cached minus live top-1 exceeds \u22120.50 percentage points on the primary pair. Reuse this tolerance only for that secondary interpretation. Charge teacher training explicitly for a greenfield deployment scenario and for comparison with hard labels. The same teacher cost cancels in a like-for-like comparison of two KD alternatives. Neither the teacher nor its cache is required for student inference. An accuracy difference between identically shaped students cannot establish an inference-energy saving.\n", "cache_preflight.py": "\"\"\"Disposable checks for the KD view-allocation study. No benchmark data or fitted weights.\"\"\"\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport tempfile\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\n\n# cuBLAS reads this at initialization in a fresh process or notebook kernel.\nos.environ.setdefault(\"CUBLAS_WORKSPACE_CONFIG\", \":4096:8\")\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nPOLICIES = (\"uniform\", \"shuffled\", \"entropy\", \"disagreement\")\n\n\ndef key(*fields):\n    \"\"\"Stable, domain-separated 64-bit seed for a local random generator.\"\"\"\n    raw = json.dumps(fields, ensure_ascii=True, separators=(\",\", \":\")).encode(\"ascii\")\n    return int.from_bytes(hashlib.sha256(raw).digest()[:8], \"little\")\n\n\ndef view_codes(image_id, seed, padding):\n    count = 2 * (2 * padding + 1) ** 2\n    rng = np.random.Generator(np.random.PCG64(key(\"view-bank\", seed, image_id)))\n    return rng.permutation(count)[:6].tolist()\n\n\ndef render_view(image, code, padding):\n    \"\"\"Decode one crop and flip from CHW RGB. The caller owns normalization.\"\"\"\n    width = 2 * padding + 1\n    crop, flip = divmod(code, 2)\n    top, left = divmod(crop, width)\n    if not (0 <= top < width):\n        raise ValueError(\"Invalid crop code\")\n    height, size = image.shape[-2:]\n    padded = F.pad(image.unsqueeze(0), (padding,) * 4, mode=\"reflect\")[0]\n    view = padded[:, top:top + height, left:left + size]\n    return view.flip(-1) if flip else view\n\n\ndef pilot_scores(logits):\n    \"\"\"Two-view Jensen-Shannon divergence and mean individual entropy in nats.\"\"\"\n    if logits.ndim != 3 or logits.shape[1] != 2 or not torch.isfinite(logits).all():\n        raise ValueError(\"Expected finite N by 2 by C pilot logits\")\n    logp = logits.double().log_softmax(-1)\n    logm = torch.logaddexp(logp[:, 0], logp[:, 1]) - math.log(2)\n    prob = logp.exp()\n    entropy = -(prob * logp).sum(-1).mean(1)\n    js = (prob * (logp - logm[:, None, :])).sum(-1).mean(1)\n    return js.cpu().numpy(), entropy.cpu().numpy()\n\n\ndef allocate(ids, labels, scores, policy, seed):\n    \"\"\"For an odd class, its median receives four views to preserve the exact budget.\"\"\"\n    if policy not in POLICIES or len(set(ids)) != len(ids):\n        raise ValueError(\"Unknown allocation or duplicate image IDs\")\n    if len(ids) != len(labels) or len(ids) != len(scores) or not np.isfinite(scores).all():\n        raise ValueError(\"Invalid allocation inputs\")\n    counts = {image_id: 4 for image_id in ids}\n    if policy == \"uniform\":\n        return counts\n    for label in sorted(set(labels)):\n        members = [i for i, value in enumerate(labels) if value == label]\n        if policy == \"shuffled\":\n            members.sort(key=lambda i: (key(\"random-high\", seed, ids[i]), ids[i]))\n        else:\n            members.sort(key=lambda i: (-float(scores[i]), key(\"tie\", seed, ids[i]), ids[i]))\n        half = len(members) // 2\n        for rank, index in enumerate(members):\n            counts[ids[index]] = 6 if rank < half else (2 if rank >= len(members) - half else 4)\n    return counts\n\n\ndef slot_for_epoch(image_id, seed, epoch, count):\n    return (key(\"slot-offset\", seed, image_id) + epoch) % count\n\n\ndef kd_loss(student, teacher, labels, temperature=4.0, weight=0.5):\n    ce = F.cross_entropy(student, labels)\n    kl = F.kl_div((student / temperature).log_softmax(-1),\n                  (teacher.detach() / temperature).softmax(-1), reduction=\"batchmean\")\n    return (1 - weight) * ce + weight * temperature ** 2 * kl\n\n\ndef write_csv(path, rows):\n    with path.open(\"w\", encoding=\"utf-8\", newline=\"\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef digest(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef require(condition, message):\n    if not condition:\n        raise RuntimeError(message)\n\n\ndef exercise_device(device, output):\n    output.mkdir()\n    seed = 42\n    # An odd class exercises the median case alongside two even classes.\n    labels = [0] * 8 + [1] * 8 + [2] * 9\n    ids = [f\"fit/{i:03d}\" for i in range(len(labels))]\n    held_out = {\"validation/000\", \"test/000\"}\n    rng = np.random.default_rng(794)\n    images = {i: torch.from_numpy(rng.integers(0, 256, (3, 16, 16), dtype=np.uint8))\n              for i in ids}\n    codes = {i: view_codes(i, seed, padding=2) for i in ids}\n    require(all(len(set(values)) == 6 for values in codes.values()), \"Repeated view key\")\n    torch.manual_seed(827)\n    teacher = nn.Sequential(nn.Conv2d(3, 8, 3), nn.BatchNorm2d(8), nn.ReLU(),\n                            nn.AdaptiveAvgPool2d((2, 2)), nn.Flatten(), nn.Linear(32, 3))\n    teacher = teacher.to(device).eval().requires_grad_(False)\n    student = nn.Sequential(nn.Conv2d(3, 5, 3), nn.ReLU(), nn.AdaptiveAvgPool2d(1),\n                            nn.Flatten(), nn.Linear(5, 3)).to(device)\n\n    def inputs(keys):\n        return torch.stack([render_view(images[i], codes[i][j], 2).float().div(255)\n                            for i, j in keys]).to(device)\n\n    query_counts = Counter()\n\n    def query(keys, phase):\n        query_counts[phase] += len(keys)\n        with torch.no_grad():\n            return teacher(inputs(keys)).cpu()\n\n    probes = [(i, j) for i in ids for j in range(2)]\n    pilot = query(probes, \"shared_pilot\").reshape(len(ids), 2, -1)\n    js, entropy = pilot_scores(pilot)\n    equal_js, _ = pilot_scores(pilot[:, :1].expand(-1, 2, -1))\n    require(np.max(np.abs(equal_js)) < 1e-12, \"Identical probe distributions disagreed\")\n    require(np.all((js >= -1e-12) & (js <= math.log(2) + 1e-12)), \"JSD out of range\")\n    scores = [{\"image_id\": i, \"label\": y, \"js_nats\": j, \"mean_entropy_nats\": h}\n              for i, y, j, h in zip(ids, labels, js, entropy)]\n    write_csv(output / \"pilot_scores.csv\", scores)\n    stats = []\n    for policy in POLICIES:\n        counts = allocate(ids, labels, entropy if policy == \"entropy\" else js, policy, seed)\n        for label in set(labels):\n            members = [i for i, y in zip(ids, labels) if y == label]\n            require(sum(counts[i] for i in members) == 4 * len(members), \"Class budget mismatch\")\n        extra = [(i, j) for i in ids for j in range(2, counts[i])]\n        keys = probes + extra\n        require(len(set(keys)) == 4 * len(ids), \"Duplicate or missing cache rows\")\n        require({i for i, _ in keys} == set(ids) and held_out.isdisjoint(i for i, _ in keys),\n                \"Non-fitting ID in cache\")\n        values = torch.cat([pilot.reshape(len(probes), -1), query(extra, policy)]).numpy()\n        charged = query_counts[\"shared_pilot\"] + query_counts[policy]\n        require(charged == 4 * len(ids), \"Teacher call ledger differs from cache budget\")\n        bank = output / f\"{policy}_logits.npy\"\n        index = output / f\"{policy}_views.csv\"\n        np.save(bank, values, allow_pickle=False)\n        write_csv(index, [{\"row\": r, \"image_id\": i, \"slot\": j, \"view_code\": codes[i][j]}\n                          for r, (i, j) in enumerate(keys)])\n        with index.open(encoding=\"utf-8\", newline=\"\") as f:\n            restored_index = list(csv.DictReader(f))\n        lookup = {(row[\"image_id\"], int(row[\"slot\"])): int(row[\"row\"]) for row in restored_index}\n        cache = np.load(bank, allow_pickle=False)\n        require(len(lookup) == len(cache) == len(keys), \"Index/body mismatch\")\n        # Rebuild every stored view independently from the persisted crop code.\n        replay_images = torch.stack([\n            render_view(images[row[\"image_id\"]], int(row[\"view_code\"]), 2).float().div(255)\n            for row in restored_index]).to(device)\n        with torch.no_grad():\n            live_all = teacher(replay_images).cpu().numpy()\n        logit_error = float(np.max(np.abs(live_all - cache)))\n        sample_indices = list(range(0, 24, 3))\n        selected = [(ids[k], slot_for_epoch(ids[k], seed, 17, counts[ids[k]])) for k in sample_indices]\n        cached = torch.from_numpy(cache[[lookup[item] for item in selected]]).to(device)\n        batch = inputs(selected)\n        target = torch.tensor([labels[k] for k in sample_indices], device=device)\n        with torch.no_grad():\n            live = teacher(batch)\n        pred = student(batch)\n        online_loss, cached_loss = kd_loss(pred, live, target), kd_loss(pred, cached, target)\n        grad_live = torch.autograd.grad(online_loss, tuple(student.parameters()), retain_graph=True)\n        grad_cache = torch.autograd.grad(cached_loss, tuple(student.parameters()))\n        grad_error = max((a - b).abs().max().item() for a, b in zip(grad_live, grad_cache))\n        loss_error = abs(online_loss.item() - cached_loss.item())\n        require(logit_error <= 1e-5 and loss_error <= 1e-6 and grad_error <= 1e-5,\n                f\"Replay mismatch for {policy}: logits {logit_error}, loss {loss_error}, grad {grad_error}\")\n        require(all(p.grad is None for p in teacher.parameters()), \"Teacher gradient\")\n        for i in ids:\n            exposure = Counter(slot_for_epoch(i, seed, e, counts[i]) for e in range(60))\n            require(len(exposure) == counts[i] and set(exposure.values()) == {60 // counts[i]},\n                    \"Unequal slot exposure across the planned 60 epochs\")\n        stats.append(dict(policy=policy, fit_images=len(ids), pilot_queries=query_counts[\"shared_pilot\"],\n                          additional_queries=query_counts[policy], standalone_queries=charged,\n                          replay_audit_queries=len(restored_index) + len(selected),\n                          logit_payload_bytes=cache.nbytes, index_bytes=index.stat().st_size,\n                          max_logit_error=logit_error, loss_error=loss_error,\n                          max_gradient_error=grad_error, bank_sha256=digest(bank),\n                          index_sha256=digest(index)))\n    write_csv(output / \"policy_checks.csv\", stats)\n    return stats\n\n\ndef run_fixture(output_parent):\n    \"\"\"Make a new evidence directory on every invocation and preserve previous receipts.\"\"\"\n    torch.set_num_threads(1)\n    torch.use_deterministic_algorithms(True)\n    if torch.cuda.is_available():\n        torch.backends.cudnn.benchmark = False\n        torch.backends.cudnn.allow_tf32 = False\n        torch.backends.cuda.matmul.allow_tf32 = False\n    output_parent = Path(output_parent)\n    output_parent.mkdir(parents=True, exist_ok=True)\n    target = Path(tempfile.mkdtemp(prefix=\"cache-fixture-\", dir=output_parent))\n    devices = [torch.device(\"cpu\")] + [torch.device(\"cuda\", n) for n in range(torch.cuda.device_count())]\n    results = {}\n    for device in devices:\n        tag = str(device).replace(\":\", \"-\")\n        results[tag] = exercise_device(device, target / tag)\n    receipt = dict(kind=\"synthetic engineering fixture\", torch=torch.__version__,\n                   numpy=np.__version__, python=platform.python_version(), devices=[\n                       torch.cuda.get_device_name(n) for n in range(torch.cuda.device_count())\n                   ], results=results)\n    (target / \"receipt.json\").write_text(json.dumps(receipt, indent=2), encoding=\"utf-8\")\n    return target, receipt\n", "cifar100_manifest.py": "\"\"\"Build unlabeled evaluation role manifests from the official CIFAR-100 binary archive.\"\"\"\n\nimport csv\nimport hashlib\nimport json\nimport tarfile\nimport tempfile\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\n\nARCHIVE_MD5 = \"03b5dce01913d631647c71ecec9e9cb8\"\nRECORD_BYTES = 2 + 3 * 32 * 32\n\n\ndef split_rank(image_id):\n    return hashlib.sha256(f\"kd-view-v1|1729|{image_id}\".encode(\"ascii\")).digest()\n\n\ndef make_roles(train_labels, train_hashes, test_hashes, fit_per_class):\n    if len(train_hashes) != len(train_labels):\n        raise ValueError(\"Training labels and pixel digests differ in length\")\n    members = defaultdict(list)\n    train_ids = [f\"train/{i:05d}\" for i in range(len(train_labels))]\n    for i, label in enumerate(train_labels):\n        members[int(label)].append(i)\n    fit = set()\n    for indices in members.values():\n        if len(indices) <= fit_per_class:\n            raise ValueError(\"Class lacks sufficient validation candidates\")\n        ordered = sorted(indices, key=lambda i: (split_rank(train_ids[i]), train_ids[i]))\n        fit.update(ordered[:fit_per_class])\n    seen_fit = {train_hashes[i] for i in fit}\n    seen_val = set()\n    rows = []\n    for i, (image_id, label, pixel_sha) in enumerate(zip(train_ids, train_labels, train_hashes)):\n        if i in fit:\n            role, reason = \"fit\", \"\"\n        elif pixel_sha in seen_fit:\n            role, reason = \"val_excluded\", \"matches_fit\"\n        elif pixel_sha in seen_val:\n            role, reason = \"val_excluded\", \"repeated_validation_pixels\"\n        else:\n            role, reason = \"validation\", \"\"\n            seen_val.add(pixel_sha)\n        rows.append(dict(image_id=image_id, role=role, fine_label=int(label),\n                         rgb_chw_sha256=pixel_sha, exclusion=reason))\n    all_train = set(train_hashes)\n    seen_test = set()\n    for i, pixel_sha in enumerate(test_hashes):\n        if pixel_sha in all_train:\n            reason = \"matches_official_training\"\n        elif pixel_sha in seen_test:\n            reason = \"repeated_test_pixels\"\n        else:\n            reason = \"\"\n        seen_test.add(pixel_sha)\n        rows.append(dict(image_id=f\"test/{i:05d}\", role=\"test_excluded\" if reason else \"test_primary\",\n                         fine_label=\"\", rgb_chw_sha256=pixel_sha, exclusion=reason))\n    return rows\n\n\ndef check_split_fixture():\n    labels = [0] * 4 + [1] * 4\n    train_ids = [f\"train/{i:05d}\" for i in range(8)]\n    by_rank = sorted(range(4), key=lambda i: split_rank(train_ids[i]))\n    hashes = [str(i) for i in range(8)]\n    hashes[by_rank[-1]] = hashes[by_rank[0]]\n    rows = make_roles(labels, hashes, [hashes[by_rank[0]], \"new\", \"new\", \"other\"], 2)\n    counts = Counter(r[\"role\"] for r in rows)\n    expected = {\"fit\": 4, \"validation\": 3, \"val_excluded\": 1, \"test_primary\": 2, \"test_excluded\": 2}\n    if counts != expected or any(r[\"fine_label\"] != \"\" for r in rows if r[\"image_id\"].startswith(\"test/\")):\n        raise RuntimeError(\"Split fixture failed\")\n    return dict(counts)\n\n\ndef audit_cifar100(archive_path, output_parent):\n    archive_path = Path(archive_path)\n    md5, sha = hashlib.md5(), hashlib.sha256()\n    with archive_path.open(\"rb\") as file:\n        for data in iter(lambda: file.read(1024 * 1024), b\"\"):\n            md5.update(data)\n            sha.update(data)\n    if md5.hexdigest() != ARCHIVE_MD5:\n        raise ValueError(\"Archive differs from the upstream CIFAR-100 binary edition\")\n    with tarfile.open(archive_path, \"r:gz\") as tar:\n        def read_split(name, expected):\n            entry = tar.getmember(f\"cifar-100-binary/{name}.bin\")\n            if not entry.isfile() or entry.size != expected * RECORD_BYTES:\n                raise ValueError(f\"Unexpected {name} binary data\")\n            with tar.extractfile(entry) as stream:\n                raw = np.frombuffer(stream.read(), dtype=np.uint8).reshape(expected, RECORD_BYTES)\n            labels = raw[:, 1].astype(int).tolist() if name == \"train\" else []\n            hashes = [hashlib.sha256(row[2:]).hexdigest() for row in raw]\n            return labels, hashes\n        labels, train_hashes = read_split(\"train\", 50000)\n        _, test_hashes = read_split(\"test\", 10000)\n    if Counter(labels) != {i: 500 for i in range(100)}:\n        raise ValueError(\"Unexpected fine-class support\")\n    rows = make_roles(labels, train_hashes, test_hashes, fit_per_class=450)\n    counts = Counter(row[\"role\"] for row in rows)\n    if len(rows) != 60000 or counts[\"fit\"] != 45000:\n        raise RuntimeError(\"Split size mismatch\")\n    root = Path(output_parent)\n    root.mkdir(parents=True, exist_ok=True)\n    out = Path(tempfile.mkdtemp(prefix=\"cifar100-manifest-\", dir=root))\n    csv_path = out / \"split_manifest.csv\"\n    with csv_path.open(\"w\", encoding=\"utf-8\", newline=\"\") as f:\n        w = csv.DictWriter(f, fieldnames=list(rows[0]))\n        w.writeheader()\n        w.writerows(rows)\n    result = dict(archive_md5=md5.hexdigest(), archive_sha256=sha.hexdigest(),\n                  manifest_sha256=hashlib.sha256(csv_path.read_bytes()).hexdigest(),\n                  counts=dict(counts), test_labels_exported=False)\n    (out / \"split_receipt.json\").write_text(json.dumps(result, indent=2), encoding=\"utf-8\")\n    return out, result\n", "kd_campaign.py": "\"\"\"Sequential CIFAR-100 teacher and six-arm runner for a reserved Kaggle session.\"\"\"\n\nimport csv\nimport hashlib\nimport os\nimport shutil\nimport signal\nimport sys\nimport tempfile\nimport time\nimport uuid\nfrom pathlib import Path\n\nos.environ.setdefault(\"CUBLAS_WORKSPACE_CONFIG\", \":4096:8\")\nimport numpy as np\nimport torch\nfrom filelock import FileLock\n\nfrom kd_fit_data import build_bank, official_images\nfrom kd_fit_engine import model_for, train_fit\nfrom kd_review import audit_fit, final_release\nfrom kd_fit_io import Checkpoints, csv_write, file_info, hash_object, json_read, json_write\nfrom kd_hub import Hub\nfrom kd_meter import BoardWindow\nfrom kd_qualify import child, qualification, unit, verify_continuation\n\nARMS = (\"hard_labels\", \"live_kd\", \"uniform\", \"shuffled\", \"entropy\", \"disagreement\")\nSEEDS = (42, 123, 2026, 31415, 27182)\nRESERVE = 2_000_000_000\n\n\ndef environment():\n    import importlib.metadata as md\n\n    return dict(\n        python=sys.version.split()[0],\n        torch=str(torch.__version__),\n        numpy=np.__version__,\n        torchvision=md.version(\"torchvision\"),\n        cuda=torch.version.cuda,\n        cudnn=torch.backends.cudnn.version(),\n        gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n        capability=[list(torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())],\n    )\n\n\ndef qualified_source():\n    root = Path(__file__).resolve().parent\n    paths = [\n        \"cache_preflight.py\",\n        \"cifar100_manifest.py\",\n        \"kd_fit_io.py\",\n        \"kd_fit_data.py\",\n        \"kd_fit_engine.py\",\n        \"kd_hub.py\",\n        \"kd_meter.py\",\n        \"kd_campaign.py\",\n        \"kd_qualify.py\",\n        \"kd_review.py\",\n    ]\n    return hash_object({n: file_info(root / n)[\"sha256\"] for n in paths})\n\n\ndef footprint_probe(data, base, model_name, epochs, device):\n    cfg = dict(\n        seed=119,\n        arm=\"teacher\" if model_name == \"resnet34\" else \"live_kd\",\n        epochs=epochs,\n        batch=128,\n        warm_epochs=5,\n        commit_updates=32,\n        model=model_name,\n    )\n    teacher = model_for(\"resnet34\", data.classes).to(device).eval() if cfg[\"arm\"] == \"live_kd\" else None\n    store = Checkpoints(\n        base, {\"fixture\": \"capacity\", \"cfg\": cfg, \"data\": data.identity, \"environment\": environment()}\n    )\n    steps = (len(data.fit) + 127) // 128\n    start = time.perf_counter()\n    state = train_fit(store, data, cfg, device, lambda *_: None, teacher=teacher, until=steps)\n    elapsed = time.perf_counter() - start\n    paths = store.pointer[\"files\"]\n    if state[\"update\"] != steps:\n        raise RuntimeError(\"The capacity probe must cover one full fitting epoch\")\n    families = {}\n    step_times = []\n    for name, metadata in paths.items():\n        family = name.split(\"/\", 1)[0]\n        totals = families.setdefault(family, {\"files\": 0, \"bytes\": 0})\n        totals[\"files\"] += 1\n        totals[\"bytes\"] += metadata[\"bytes\"]\n        if name.endswith(\"/updates.csv\"):\n            with store.path(name).open(encoding=\"utf-8\", newline=\"\") as f:\n                step_times.extend(float(row[\"seconds\"]) for row in csv.DictReader(f))\n    state_bytes = paths[store.pointer[\"state_file\"]][\"bytes\"]\n    per_epoch = sum(families[family][\"bytes\"] for family in (\"updates\", \"epochs\", \"ledger\"))\n    # Three rolling-state bodies cover the selected generation, fallback and temporary write.\n    need = int(1.20 * (epochs * per_epoch + 3 * state_bytes) + 400_000_000)\n    if len(step_times) != steps:\n        raise RuntimeError(\"Missing per-update timings in the capacity probe\")\n    seconds_per_update = float(np.median(step_times))\n    return dict(\n        model=model_name,\n        epochs=epochs,\n        observed_updates=steps,\n        observed_examples=len(data.fit),\n        observed_validation_examples=len(data.validation),\n        observed_epoch_families=families,\n        rolling_state_bytes=state_bytes,\n        required_incremental_bytes=need,\n        working_reserve=RESERVE,\n        probe_elapsed_s=elapsed,\n        measured_step_seconds=seconds_per_update,\n        compute_only_forecast_hours=seconds_per_update * steps * epochs / 3600,\n        local_epoch_inclusive_forecast_hours=elapsed * epochs / 3600,\n        full_training_updates=steps * epochs,\n        service_time_in_forecast=False,\n    )\n\n\ndef erase_verified_copies(store, manifest):\n    \"\"\"Only disposable Kaggle working copies already verified in the immutable remote revision.\"\"\"\n    if not (Path(\"/kaggle/working\").resolve() in store.root.resolve().parents):\n        return\n    keep = {\"ENDPOINT.json\", \"config.json\", \"POINTER.json\"}\n    for name, item in manifest[\"files\"].items():\n        if name in keep:\n            continue\n        path = store.path(name)\n        if path.is_file() and file_info(path) == {k: item[k] for k in (\"bytes\", \"sha256\")}:\n            path.unlink()\n    # The unselected rolling slot is superseded by the verified complete fit.\n    for name in (\"rolling/state_0.pt\", \"rolling/state_1.pt\"):\n        path = store.path(name)\n        if name not in manifest[\"files\"] and path.is_file():\n            path.unlink()\n\n\nclass Driver:\n    def __init__(self, base, hub, data, device, archives, receipt):\n        self.base, self.hub, self.data, self.device = base, hub, data, device\n        self.archives, self.receipt = archives, receipt\n        self.stopping = False\n        self.source = qualified_source()\n\n    def run_unit(self, name, config, required, work, keep=()):\n        store = Checkpoints(\n            self.base / \"units\" / name,\n            {\n                \"source\": self.source,\n                \"config\": config,\n                \"data\": self.data.identity,\n                \"environment\": environment(),\n                \"protocol_sha256\": self.receipt[\"protocol_sha256\"],\n            },\n        )\n        remote, rev = self.hub.selected(name)\n        if remote and remote[\"identity\"] != store.tag:\n            raise RuntimeError(\"Previous unit binds different data, code, teacher or environment\")\n        if remote and remote[\"complete\"]:\n            if keep:\n                self.hub.fetch(name, remote, rev, store.root, keep)\n            print(\"Verified completed unit, skipping\", name, rev, flush=True)\n            return store, remote, rev\n        self.hub.reconcile(name, store)\n        live_files = sum(p.stat().st_size for p in store.root.rglob(\"*\") if p.is_file())\n        free = shutil.disk_usage(self.base).free\n        working = Path(\"/kaggle/working\")\n        logical = (\n            sum(p.stat().st_size for p in working.rglob(\"*\") if p.is_file()) if working.is_dir() else 0\n        )\n        available = min(free, 20_000_000_000 - logical) if working.is_dir() else free\n        if available < max(0, required - live_files) + RESERVE:\n            raise RuntimeError(\n                f\"Storage projection failed for {name}. Free or quota bytes {available}, projected {required}.\"\n            )\n        setup_dir = store.root / \"setup_telemetry\"\n        idle = BoardWindow(setup_dir, \"pre_run_idle\", self.device).start()\n        time.sleep(3)\n        idle.finish()\n        warm = BoardWindow(setup_dir, \"standard_warmup\", self.device).start()\n        with torch.inference_mode():\n            example = torch.ones((768, 768), device=self.device)\n            for _ in range(20):\n                torch.mm(example, example)\n            del example\n        warm.finish()\n        last = time.monotonic()\n        window = None\n\n        def begin():\n            return BoardWindow(store.root / \"measurement\", name, self.device).start()\n\n        def publish():\n            nonlocal window, last\n            if window is not None:\n                window.finish()\n                window = None\n            if store.pointer is not None:\n                extras = {\n                    p.relative_to(store.root).as_posix(): file_info(p)\n                    for folder in (\"measurement\", \"setup_telemetry\")\n                    for p in (store.root / folder).glob(\"*.csv\")\n                }\n                self.hub.publish(name, store, extras)\n            last = time.monotonic()\n\n        def hook(current, force):\n            nonlocal window\n            if force or self.stopping or time.monotonic() - last >= 1800:\n                if current.pointer[\"complete\"] and window is not None:\n                    window.finish()\n                    window = None\n                    cooldown = BoardWindow(setup_dir, \"post_run_idle\", self.device).start()\n                    time.sleep(3)\n                    cooldown.finish()\n                publish()\n                if self.stopping:\n                    raise KeyboardInterrupt(\"Catchable stop at durable update\")\n                if not current.pointer[\"complete\"]:\n                    window = begin()\n\n        try:\n            window = begin()\n            work(store, hook)\n            if window is not None:\n                publish()\n        except BaseException:\n            try:\n                publish()\n            except Exception as transport:\n                print(\"Local files retained after publication error\", type(transport).__name__, flush=True)\n            raise\n        manifest, revision = self.hub.selected(name)\n        if not manifest or not manifest[\"complete\"] or manifest[\"chain\"] != store.pointer[\"chain\"]:\n            raise RuntimeError(\"Completed unit lacks a matching verified remote selection\")\n        return store, manifest, revision\n\n    def fit(self, config, budget, teacher=None, teacher_id=None, bank=None):\n        arm, seed = config[\"arm\"], config[\"seed\"]\n        name = (\n            f\"v1.cifar100.resnet34.teacher.{seed}\"\n            if arm == \"teacher\"\n            else f\"v1.cifar100.resnet18.{arm}.{seed}\"\n        )\n        binding = {**config, \"teacher_sha256\": teacher_id, \"bank\": bank[\"identity\"] if bank else None}\n\n        def work(store, hook):\n            if teacher is not None:\n                teacher.to(self.device)\n            try:\n                return train_fit(\n                    store, self.data, config, self.device, hook, lambda: self.stopping, teacher, bank\n                )\n            finally:\n                if teacher is not None:\n                    teacher.cpu()\n\n        store, manifest, rev = self.run_unit(name, binding, budget, work, keep=[\"ENDPOINT.json\"])\n        endpoint = json_read(store.path(\"ENDPOINT.json\"))\n        review_name = name + \"-review\"\n        review = Checkpoints(\n            self.base / \"reviews\" / name,\n            {\"fit\": store.tag, \"chain\": manifest[\"chain\"], \"review_source\": self.source},\n        )\n        old, review_rev = self.hub.selected(review_name)\n        if old and old[\"identity\"] != review.tag:\n            raise RuntimeError(\"Previous evidence audit has different fit identity\")\n        if not old or not old[\"complete\"]:\n            if store.pointer is None or store.pointer[\"chain\"] != manifest[\"chain\"]:\n                self.hub.fetch(\n                    name,\n                    manifest,\n                    rev,\n                    store.root,\n                    [n for n in manifest[\"files\"] if n != \"POINTER.json\"] + [\"POINTER.json\"],\n                )\n                store.pointer = json_read(store.pointer_path)\n            audit = audit_fit(store, self.data, bank)\n            json_write(review.path(\"fit_audit.json\"), audit)\n            review.commit(dict(update=audit[\"updates\"], complete=True), [\"fit_audit.json\"])\n            review_rev = self.hub.publish(review_name, review)\n        if arm == \"teacher\":\n            best = endpoint[\"best\"][\"weights\"]\n            self.hub.fetch(name, manifest, rev, store.root, [best])\n            result = model_for(config[\"model\"], self.data.classes).cpu().eval().requires_grad_(False)\n            result.load_state_dict(torch.load(store.path(best), map_location=\"cpu\", weights_only=True))\n            identity = manifest[\"files\"][best][\"sha256\"]\n        else:\n            result, identity = None, None\n        self.archives[name] = {\n            \"revision\": rev,\n            \"review_revision\": review_rev,\n            \"endpoint\": endpoint,\n            \"identity\": store.tag,\n        }\n        json_write(self.base / \"run_registry.json\", self.archives)\n        erase_verified_copies(store, manifest)\n        return result, identity\n\n    def bank(self, seed, arm, teacher, teacher_id):\n        name = f\"v1.cifar100.cache.{arm}.{seed}\"\n        cfg = {\"seed\": seed, \"policy\": arm, \"teacher_sha256\": teacher_id}\n\n        def work(store, hook):\n            try:\n                build_bank(store, self.data, teacher.to(self.device), self.device, seed, arm, hook)\n            finally:\n                teacher.cpu()\n\n        keep = [\"bank/codes.npy\", \"bank/counts.npy\", \"bank/logits.npy\", \"bank/scores.csv\"]\n        store, manifest, rev = self.run_unit(name, cfg, 700_000_000, work, keep=keep)\n        from cache_preflight import key\n\n        counts = np.load(store.path(keep[1]), allow_pickle=False)\n        bank = dict(\n            counts=counts,\n            offsets=np.cumsum(counts, dtype=np.int64) - counts,\n            codes=np.load(store.path(keep[0]), allow_pickle=False),\n            logits=np.load(store.path(keep[2]), mmap_mode=\"r\", allow_pickle=False),\n            slot_offsets=np.array(\n                [key(\"slot-offset\", seed, self.data.ids[i]) % int(c) for i, c in zip(self.data.fit, counts)]\n            ),\n            identity=hash_object({\"contract\": store.tag, \"logits\": manifest[\"files\"][keep[2]][\"sha256\"]}),\n        )\n        gate_name = name + \"-replay\"\n        gate = Checkpoints(\n            self.base / \"reviews\" / gate_name, {\"fit_bank\": bank[\"identity\"], \"source\": self.source}\n        )\n        previous, _ = self.hub.selected(gate_name)\n        if previous and previous[\"identity\"] != gate.tag:\n            raise RuntimeError(\"A different replay audit already exists\")\n        if not previous or not previous[\"complete\"]:\n            import copy\n            from cache_preflight import kd_loss\n\n            secondary = torch.device(\"cuda:1\")\n            anchor = copy.deepcopy(teacher).to(secondary).eval()\n            pos = np.random.default_rng(seed).choice(len(counts), 128, replace=False)\n            slots = counts[pos] - 1\n            x, y = self.data.inputs(self.data.fit[pos], bank[\"codes\"][pos, slots], secondary)\n            expected = torch.from_numpy(np.array(bank[\"logits\"][bank[\"offsets\"][pos] + slots])).to(\n                secondary\n            )\n            with torch.no_grad():\n                actual = anchor(x)\n            student = model_for(\"resnet18\", self.data.classes).to(secondary).eval()\n            z = student(x)\n            a, b = kd_loss(z, actual, y), kd_loss(z, expected, y)\n            g1 = torch.autograd.grad(a, tuple(student.parameters()), retain_graph=True)\n            g2 = torch.autograd.grad(b, tuple(student.parameters()))\n            error = dict(\n                logit=(actual - expected).abs().max().item(),\n                loss=abs(a.item() - b.item()),\n                gradient=max((i - j).abs().max().item() for i, j in zip(g1, g2)),\n                audit_queries=128,\n            )\n            if error[\"logit\"] > 1e-5 or error[\"loss\"] > 1e-6 or error[\"gradient\"] > 1e-5:\n                json_write(gate.path(\"failed_replay.json\"), error)\n                raise RuntimeError(\"Actual trained teacher failed the cross-device replay gate\")\n            json_write(gate.path(\"replay.json\"), error)\n            gate.commit({\"update\": 128, \"complete\": True}, [\"replay.json\"])\n            self.hub.publish(gate_name, gate)\n        if arm == \"disagreement\":\n            diagnostic_name = name + \"-independent-views\"\n            extra = Checkpoints(\n                self.base / \"reviews\" / diagnostic_name, {\"bank\": bank[\"identity\"], \"source\": self.source}\n            )\n            present, _ = self.hub.selected(diagnostic_name)\n            if present and present[\"identity\"] != extra.tag:\n                raise RuntimeError(\"Independent-views diagnostic identity differs\")\n            if not present or not present[\"complete\"]:\n                from cache_preflight import pilot_scores\n                from scipy.stats import spearmanr\n                from kd_review import read_csv\n\n                fitting_labels = self.data.labels[self.data.fit]\n                original = read_csv(store.path(\"bank/scores.csv\"))\n                count_codes = 2 * (2 * self.data.padding + 1) ** 2\n                records = []\n                rho = []\n                teacher.to(self.device)\n                for label in range(self.data.classes):\n                    members = np.flatnonzero(fitting_labels == label)\n                    subset = sorted(\n                        members, key=lambda pos: key(\"holdout-probe-ids\", self.data.ids[self.data.fit[pos]])\n                    )[:10]\n                    pairs = []\n                    for pos in subset:\n                        options = np.setdiff1d(np.arange(count_codes), bank[\"codes\"][pos])\n                        rng = np.random.default_rng(\n                            key(\"holdout-probe-views\", seed, self.data.ids[self.data.fit[pos]])\n                        )\n                        pairs.append(rng.choice(options, 2, replace=False))\n                    codes = np.asarray(pairs)\n                    x, _ = self.data.inputs(self.data.fit[np.repeat(subset, 2)], codes.ravel(), self.device)\n                    with torch.inference_mode():\n                        js, _ = pilot_scores(teacher(x).reshape(len(subset), 2, -1).cpu())\n                    baseline = [float(original[pos][\"js_nats\"]) for pos in subset]\n                    corr = float(spearmanr(baseline, js).statistic)\n                    rho.append(corr)\n                    records.extend(\n                        (seed, label, self.data.ids[self.data.fit[pos]], q[0], q[1], baseline[j], js[j])\n                        for j, (pos, q) in enumerate(zip(subset, pairs))\n                    )\n                csv_write(\n                    extra.path(\"independent_views.csv\"),\n                    [\n                        \"seed\",\n                        \"label\",\n                        \"image_id\",\n                        \"new_code_0\",\n                        \"new_code_1\",\n                        \"ranking_js_nats\",\n                        \"independent_js_nats\",\n                    ],\n                    records,\n                )\n                csv_write(\n                    extra.path(\"class_rank_associations.csv\"),\n                    [\"class\", \"spearman_rho\"],\n                    ((i, r if np.isfinite(r) else \"\") for i, r in enumerate(rho)),\n                )\n                extra.commit(\n                    {\"update\": 2 * len(records), \"complete\": True},\n                    [\"independent_views.csv\", \"class_rank_associations.csv\"],\n                )\n                self.hub.publish(diagnostic_name, extra)\n                teacher.cpu()\n        self.archives[name] = {\"revision\": rev, \"identity\": store.tag}\n        return bank, store, manifest\n\n\ndef run_master(mode, base, archive=None, repo=None, blocks=(42,), protocol_sha256=None):\n    base = Path(base).resolve()\n    base.mkdir(parents=True, exist_ok=True)\n    if mode not in (\"qualification\", \"measured\", \"final_review\"):\n        raise ValueError(\"Choose qualification, measured or final_review mode\")\n    if mode == \"qualification\":\n        return {\n            str(device): qualification(Path(tempfile.mkdtemp(prefix=\"local-gate-\", dir=base)), device)\n            for device in ([\"cpu\"] + [f\"cuda:{i}\" for i in range(torch.cuda.device_count())])\n        }\n    if not set(blocks).issubset(SEEDS) or len(set(blocks)) != len(blocks) or not blocks:\n        raise ValueError(\"Choose distinct preregistered seed blocks\")\n    if not archive or not repo or not protocol_sha256:\n        raise ValueError(\n            \"Measured work requires the original binary archive, a dedicated HF dataset and protocol hash\"\n        )\n    env = environment()\n    if (\n        len(env[\"gpus\"]) != 2\n        or any(\"T4\" not in n for n in env[\"gpus\"])\n        or not Path(\"/kaggle/working\").is_dir()\n    ):\n        raise RuntimeError(\"Measured qualification requires a Kaggle dual-T4 session\")\n    from kaggle_secrets import UserSecretsClient\n\n    hub = Hub(repo, UserSecretsClient().get_secret(\"HF_TOKEN\"), base / \"transport.jsonl\")\n    seedset = tuple(blocks)\n    with FileLock(str(base / \"owner.lock\"), timeout=1):\n        hub.claim()\n        driver = None\n        previous = {}\n        try:\n            data = official_images(archive, base / \"input\")\n            if mode == \"final_review\":\n                return final_release(\n                    hub, base, archive, data, ARMS, SEEDS, qualified_source(), protocol_sha256\n                )\n            gate = Path(tempfile.mkdtemp(prefix=\"target-gate-\", dir=base))\n            # Both cards explicitly perform replay/restart work. Scientific timing occupies GPU 0 alone.\n            first = qualification(gate / \"gpu0\", \"cuda:0\", all_arms=True)\n            second = qualification(gate / \"gpu1\", \"cuda:1\", all_arms=False)\n            dev = torch.device(\"cuda:0\")\n            name = \"qualification-\" + uuid.uuid4().hex\n            child(gate / \"seam_source\", dev, until=2)\n            _, _, probe = unit(gate / \"seam_source\", \"teacher\", dev)\n            rev = hub.publish(name, probe)\n            snapshot = hub.read(name + \"/MANIFEST.json\", rev)\n            recovered = gate / \"seam_download\"\n            hub.fetch(name, snapshot, rev, recovered)\n            child(recovered, dev)\n            seam = verify_continuation(gate / \"gpu0\" / \"uninterrupted\", recovered, dev)\n            forecasts = {}\n            for model, epochs in ((\"resnet34\", 90), (\"resnet18\", 60)):\n                with tempfile.TemporaryDirectory(prefix=\"capacity-\" + model + \"-\", dir=gate) as disposable:\n                    forecasts[model] = footprint_probe(data, Path(disposable), model, epochs, dev)\n            receipt = dict(\n                protocol_sha256=protocol_sha256,\n                source=qualified_source(),\n                environment=env,\n                data=data.identity,\n                first=first,\n                second=second,\n                remote_gate_revision=rev,\n                pinned_download_continuation=seam,\n                forecasts=forecasts,\n            )\n            json_write(gate / \"target_qualification.json\", receipt)\n            # Preserve the successful on-target receipts and split before any measured fit.\n            gate_store = Checkpoints(\n                gate / \"release\",\n                {\"source\": qualified_source(), \"dataset\": data.identity, \"kind\": \"qualification\"},\n            )\n            from shutil import copyfile\n\n            names = [\"target_qualification.json\", \"split_manifest.csv\", \"split_receipt.json\", \"protocol.md\"]\n            input_record = json_read(base / \"input\" / \"INPUT.json\")\n            split = base / \"input\" / input_record[\"manifest\"]\n            for src, dst in zip(\n                [\n                    gate / \"target_qualification.json\",\n                    split,\n                    split.with_name(\"split_receipt.json\"),\n                    Path(__file__).with_name(\"PROTOCOL.md\"),\n                ],\n                names,\n            ):\n                copyfile(src, gate_store.path(dst))\n            gate_store.commit(dict(update=0, complete=True), names)\n            hub.publish(name + \"-evidence\", gate_store)\n            archives = (\n                json_read(base / \"run_registry.json\") if (base / \"run_registry.json\").exists() else {}\n            )\n            driver = Driver(base, hub, data, dev, archives, receipt)\n\n            def signal_stop(sig, frame):\n                driver.stopping = True\n                print(\"Stop requested. Finishing the current complete update.\", flush=True)\n\n            for sig in (signal.SIGINT, signal.SIGTERM):\n                previous[sig] = signal.signal(sig, signal_stop)\n            teacher_cfg = dict(\n                seed=907,\n                arm=\"teacher\",\n                model=\"resnet34\",\n                epochs=90,\n                batch=128,\n                warm_epochs=5,\n                commit_updates=32,\n            )\n            teacher, teacher_id = driver.fit(\n                teacher_cfg, forecasts[\"resnet34\"][\"required_incremental_bytes\"]\n            )\n            for seed in seedset:\n                order = sorted(\n                    ARMS,\n                    key=lambda a: hashlib.sha256(\n                        f\"order-v1|cifar100|resnet18|{seed}|{a}\".encode()\n                    ).digest(),\n                )\n                for arm in order:\n                    cfg = dict(\n                        seed=seed,\n                        arm=arm,\n                        model=\"resnet18\",\n                        epochs=60,\n                        batch=128,\n                        warm_epochs=5,\n                        commit_updates=32,\n                    )\n                    bank = bank_store = bank_manifest = None\n                    if arm not in (\"hard_labels\", \"live_kd\"):\n                        bank, bank_store, bank_manifest = driver.bank(seed, arm, teacher, teacher_id)\n                    driver.fit(\n                        cfg,\n                        forecasts[\"resnet18\"][\"required_incremental_bytes\"],\n                        teacher if arm == \"live_kd\" else None,\n                        teacher_id if arm != \"hard_labels\" else None,\n                        bank,\n                    )\n                    if bank_store:\n                        del bank\n                        erase_verified_copies(bank_store, bank_manifest)\n            return {\n                \"qualified\": True,\n                \"data_sha256\": data.identity,\n                \"registry\": driver.archives,\n                \"test_release\": \"sealed\",\n                \"qualification_revision\": rev,\n            }\n        finally:\n            for sig, handler in previous.items():\n                signal.signal(sig, handler)\n            hub.release()\n", "kd_fit_data.py": "\"\"\"CIFAR-100 training cohort, deterministic replay and quota-matched teacher banks.\"\"\"\n\nimport csv\nimport tarfile\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nfrom cache_preflight import allocate, key, pilot_scores, view_codes\nfrom cifar100_manifest import RECORD_BYTES, audit_cifar100\nfrom kd_fit_io import csv_write, file_info, hash_object, json_read, json_write, np_write\n\n\nclass Images:\n    def __init__(self, rgb, labels, ids, fit, validation, classes, identity, padding):\n        self.rgb = rgb\n        self.labels = np.asarray(labels, dtype=np.int64)\n        self.ids = list(ids)\n        self.fit = np.asarray(fit, dtype=np.int64)\n        self.validation = np.asarray(validation, dtype=np.int64)\n        self.classes, self.identity, self.padding = classes, identity, padding\n        total = np.zeros(3, dtype=np.float64)\n        squared = total.copy()\n        for start in range(0, len(self.fit), 256):\n            a = self.rgb[self.fit[start : start + 256]].astype(np.float64) / 255\n            total += a.sum(axis=(0, 2, 3))\n            squared += (a * a).sum(axis=(0, 2, 3))\n        n = len(self.fit) * rgb.shape[2] * rgb.shape[3]\n        self.mean = total / n\n        self.std = np.sqrt(squared / n - self.mean**2)\n        if not np.all(self.std > 0):\n            raise ValueError(\"A fitting channel has zero variance\")\n\n    def inputs(self, indices, codes, device):\n        a = torch.from_numpy(self.rgb[indices]).to(device)\n        if codes is not None:\n            size, pad = a.shape[-1], self.padding\n            flags = torch.as_tensor(codes, dtype=torch.long, device=device)\n            crop, flip = flags // 2, flags % 2\n            top, left = crop // (2 * pad + 1), crop % (2 * pad + 1)\n            x = torch.arange(size, device=device)\n            cols = torch.where(flip[:, None].bool(), size - 1 - x, x) + left[:, None]\n            pos = (top[:, None, None] + x[None, :, None]) * (size + 2 * pad) + cols[:, None, :]\n            padded = F.pad(a, (pad,) * 4, mode=\"reflect\").flatten(2)\n            a = padded.gather(2, pos.reshape(len(indices), 1, -1).expand(-1, 3, -1)).reshape_as(a)\n        a = a.float().div_(255)\n        mu = torch.as_tensor(self.mean, device=device, dtype=a.dtype)[None, :, None, None]\n        sd = torch.as_tensor(self.std, device=device, dtype=a.dtype)[None, :, None, None]\n        return (a - mu) / sd, torch.as_tensor(self.labels[indices], device=device)\n\n\ndef official_images(archive, root):\n    \"\"\"Source archive is hashed every launch. Teacher/student access only the official training member.\"\"\"\n    archive, root = Path(archive), Path(root)\n    meta = file_info(archive)\n    index_file = root / \"INPUT.json\"\n    root.mkdir(parents=True, exist_ok=True)\n    if index_file.exists():\n        saved = json_read(index_file)\n        path = root / saved[\"manifest\"]\n        if meta != saved[\"archive\"] or file_info(path)[\"sha256\"] != saved[\"split\"][\"manifest_sha256\"]:\n            raise RuntimeError(\"Original archive or frozen split differs\")\n    else:\n        out, receipt = audit_cifar100(archive, root)\n        path = out / \"split_manifest.csv\"\n        saved = dict(archive=meta, split=receipt, manifest=path.relative_to(root).as_posix())\n        json_write(index_file, saved)\n    with tarfile.open(archive, \"r:gz\") as tar:\n        with tar.extractfile(\"cifar-100-binary/train.bin\") as stream:\n            raw = np.frombuffer(stream.read(), dtype=np.uint8).reshape(50000, RECORD_BYTES)\n    with path.open(encoding=\"utf-8\", newline=\"\") as f:\n        rows = [r for r in csv.DictReader(f) if r[\"image_id\"].startswith(\"train/\")]\n    ids = [r[\"image_id\"] for r in rows]\n    if len(rows) != len(raw) or ids != [f\"train/{i:05d}\" for i in range(50000)]:\n        raise ValueError(\"Source order mismatch\")\n    if not np.array_equal(np.asarray([int(r[\"fine_label\"]) for r in rows]), raw[:, 1]):\n        raise ValueError(\"Source label mismatch\")\n    return Images(\n        raw[:, 2:].reshape(-1, 3, 32, 32),\n        raw[:, 1],\n        ids,\n        [i for i, r in enumerate(rows) if r[\"role\"] == \"fit\"],\n        [i for i, r in enumerate(rows) if r[\"role\"] == \"validation\"],\n        100,\n        saved[\"split\"][\"manifest_sha256\"],\n        4,\n    )\n\n\ndef toy_images():\n    rng = np.random.default_rng(311)\n    a = rng.integers(0, 256, (32, 3, 12, 12), dtype=np.uint8)\n    y = np.arange(32) % 4\n    from hashlib import sha256\n\n    return Images(\n        a,\n        y,\n        [f\"toy/{i:02d}\" for i in range(32)],\n        np.arange(24),\n        np.arange(24, 32),\n        4,\n        sha256(a.tobytes() + y.tobytes()).hexdigest(),\n        2,\n    )\n\n\ndef source_order(n, seed, epoch):\n    return np.random.default_rng(key(\"source-order\", seed, epoch)).permutation(n)\n\n\ndef fresh_codes(data, seed, epoch):\n    rng = np.random.default_rng(key(\"fresh\", seed, epoch))\n    return rng.integers(0, 2 * (2 * data.padding + 1) ** 2, size=len(data.fit))\n\n\ndef build_bank(store, data, teacher, device, seed, policy, checkpoint_hook, group=4096):\n    \"\"\"Each phase is chunked. Progress is committed before the periodic publication hook.\"\"\"\n    state = store.load() or {\"phase\": \"probes\", \"position\": 0, \"update\": 0, \"complete\": False}\n    ids = [data.ids[i] for i in data.fit]\n    labels = data.labels[data.fit].tolist()\n    codes = np.asarray([view_codes(i, seed, data.padding) for i in ids], dtype=np.int16)\n    n = len(ids)\n    teacher.eval().requires_grad_(False)\n\n    def query_chunks(phase, positions, slots):\n        for start in range(state[\"position\"], len(positions), group):\n            end = min(start + group, len(positions))\n            pieces = []\n            with torch.inference_mode():\n                for first in range(start, end, 128):\n                    last = min(first + 128, end)\n                    ii, jj = positions[first:last], slots[first:last]\n                    x, _ = data.inputs(data.fit[ii], codes[ii, jj], device)\n                    pieces.append(teacher(x).cpu().numpy())\n            name = f\"cache/{phase}_{start:08d}.npy\"\n            out = np.concatenate(pieces).astype(np.float32, copy=False)\n            if not np.isfinite(out).all():\n                raise ValueError(\"Nonfinite cache logits\")\n            np_write(store.path(name), out)\n            state.update(position=end, update=state[\"update\"] + end - start)\n            store.commit(state, [name])\n            checkpoint_hook(store, False)\n\n    def read_chunks(phase):\n        names = sorted(k for k in store.pointer[\"files\"] if k.startswith(f\"cache/{phase}_\"))\n        return np.concatenate([np.load(store.path(name), allow_pickle=False) for name in names])\n\n    if not state[\"complete\"]:\n        probe_i, probe_j = np.repeat(np.arange(n), 2), np.tile(np.arange(2), n)\n        if state[\"phase\"] == \"probes\":\n            query_chunks(\"probe\", probe_i, probe_j)\n            state.update(phase=\"extra\", position=0)\n            store.commit(state, [])\n        probes = read_chunks(\"probe\").reshape(n, 2, data.classes)\n        js, entropy = pilot_scores(torch.from_numpy(probes))\n        quotas = allocate(ids, labels, entropy if policy == \"entropy\" else js, policy, seed)\n        counts = np.asarray([quotas[i] for i in ids], dtype=np.int16)\n        extra_i = np.repeat(np.arange(n), counts - 2)\n        extra_j = np.concatenate([np.arange(2, c) for c in counts if c > 2])\n        query_chunks(\"extra\", extra_i, extra_j)\n        extras = read_chunks(\"extra\")\n        offsets = np.cumsum(counts, dtype=np.int64) - counts\n        packed = np.empty((4 * n, data.classes), dtype=np.float32)\n        packed[offsets] = probes[:, 0]\n        packed[offsets + 1] = probes[:, 1]\n        packed[offsets[extra_i] + extra_j] = extras\n        if counts.sum() != 4 * n or state[\"update\"] != 4 * n:\n            raise RuntimeError(\"Teacher calls and target row budget differ\")\n        np_write(store.path(\"bank/logits.npy\"), packed)\n        np_write(store.path(\"bank/counts.npy\"), counts)\n        np_write(store.path(\"bank/codes.npy\"), codes)\n        csv_write(\n            store.path(\"bank/scores.csv\"),\n            [\"image_id\", \"fine_label\", \"js_nats\", \"entropy_nats\", \"correct_probes\", \"view_count\"],\n            (\n                (\n                    ids[i],\n                    labels[i],\n                    js[i],\n                    entropy[i],\n                    int((probes[i].argmax(-1) == labels[i]).sum()),\n                    counts[i],\n                )\n                for i in range(n)\n            ),\n        )\n        csv_write(\n            store.path(\"bank/views.csv\"),\n            [\"row\", \"image_id\", \"slot\", \"crop_flip_code\"],\n            ((offsets[i] + s, ids[i], s, codes[i, s]) for i in range(n) for s in range(counts[i])),\n        )\n        state.update(phase=\"complete\", position=0, complete=True)\n        store.commit(\n            state,\n            [\"bank/\" + n for n in (\"logits.npy\", \"counts.npy\", \"codes.npy\", \"scores.csv\", \"views.csv\")],\n        )\n        checkpoint_hook(store, True)\n    counts = np.load(store.path(\"bank/counts.npy\"), allow_pickle=False)\n    return dict(\n        counts=counts,\n        offsets=np.cumsum(counts, dtype=np.int64) - counts,\n        codes=np.load(store.path(\"bank/codes.npy\"), allow_pickle=False),\n        logits=np.load(store.path(\"bank/logits.npy\"), mmap_mode=\"r\", allow_pickle=False),\n        slot_offsets=np.array([key(\"slot-offset\", seed, i) % int(c) for i, c in zip(ids, counts)]),\n        identity=hash_object({\"contract\": store.tag, \"logits\": store.pointer[\"files\"][\"bank/logits.npy\"]}),\n    )\n", "kd_fit_engine.py": "\"\"\"Fixed-schedule training shared by the CIFAR teacher and its six student arms.\"\"\"\n\nimport math\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom kd_fit_data import fresh_codes, source_order\nfrom kd_fit_io import csv_write, json_write, np_write, rng_restore, rng_state, seed_all, torch_write\n\n\ndef model_for(name, classes):\n    if name == \"fixture\":\n        return nn.Sequential(\n            nn.Conv2d(3, 8, 3, padding=1),\n            nn.BatchNorm2d(8),\n            nn.ReLU(),\n            nn.AdaptiveAvgPool2d(1),\n            nn.Flatten(),\n            nn.Dropout(0.2),\n            nn.Linear(8, classes),\n        )\n    if os.name == \"nt\":\n        dlls = Path(sys.prefix) / \"Library/bin\"\n        if dlls.exists():\n            global _dll_handle\n            _dll_handle = os.add_dll_directory(str(dlls))\n        from PIL import Image  # noqa: F401 - load the Windows imaging DLL before torchvision.\n    from torchvision import models\n\n    if name not in (\"resnet18\", \"resnet34\"):\n        raise ValueError(\"Unsupported CIFAR-100 primary model\")\n    net = getattr(models, name)(weights=None, num_classes=classes)\n    net.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)\n    net.maxpool = nn.Identity()\n    return net\n\n\ndef lr_for(update, total, warmup):\n    if update < warmup:\n        return 0.1 * (update + 1) / warmup\n    return 0.05 * (1.0 + math.cos(math.pi * (update - warmup) / max(total - warmup - 1, 1)))\n\n\ndef evaluate(model, data, ids, device, batch):\n    model.eval()\n    chunks = []\n    with torch.inference_mode():\n        for start in range(0, len(ids), batch):\n            idx = ids[start : start + batch]\n            features, _ = data.inputs(idx, None, device)\n            chunks.append(model(features).cpu().numpy())\n    logits = np.concatenate(chunks)\n    labels = data.labels[ids]\n    if not np.isfinite(logits).all() or not len(labels):\n        raise ValueError(\"Invalid evaluation predictions or empty scoring cohort\")\n    nll = F.cross_entropy(torch.from_numpy(logits).double(), torch.from_numpy(labels)).item()\n    return logits, int(np.sum(logits.argmax(1) == labels)), nll\n\n\ndef train_fit(\n    store,\n    data,\n    config,\n    device,\n    on_commit,\n    stop_requested=lambda: False,\n    teacher=None,\n    bank=None,\n    until=None,\n    fault_update=None,\n):\n    seed, arm, epochs, batch = (config[k] for k in (\"seed\", \"arm\", \"epochs\", \"batch\"))\n    seed_all(seed)\n    model = model_for(config[\"model\"], data.classes).to(device)\n    optimizer = torch.optim.SGD(\n        model.parameters(), lr=0.1, momentum=0.9, weight_decay=0.0005, nesterov=False, foreach=False\n    )\n    state = store.load()\n    steps = math.ceil(len(data.fit) / batch)\n    total = epochs * steps\n    warm = config[\"warm_epochs\"] * steps\n    if state:\n        model.load_state_dict(state[\"model\"])\n        optimizer.load_state_dict(state[\"optimizer\"])\n        rng_restore(state[\"rng\"])\n        step, best, partial = state[\"update\"], state[\"best\"], state[\"partial\"]\n        if state[\"complete\"]:\n            return state\n    else:\n        step, best, partial = 0, None, [0, 0.0, 0.0, 0.0, 0]\n        json_write(\n            store.path(\"config.json\"),\n            {\n                **config,\n                \"contract\": store.identity,\n                \"mean\": data.mean.tolist(),\n                \"std\": data.std.tolist(),\n                \"optimizer\": \"SGD\",\n                \"scaler\": None,\n                \"learning_rate\": \"linear warm-up then cosine by optimizer update\",\n            },\n        )\n        initial = dict(\n            update=0,\n            complete=False,\n            model=model.state_dict(),\n            optimizer=optimizer.state_dict(),\n            rng=rng_state(),\n            best=None,\n            partial=partial,\n            scheduler_update=0,\n            scaler=None,\n        )\n        store.commit(initial, [\"config.json\"])\n        on_commit(store, True)\n    if arm == \"live_kd\" and teacher is None:\n        raise ValueError(\"Live KD needs the selected teacher\")\n    if arm not in (\"teacher\", \"hard_labels\", \"live_kd\") and bank is None:\n        raise ValueError(\"Cached KD needs a completed matching bank\")\n    if teacher is not None:\n        teacher.eval().requires_grad_(False)\n    exposure, update_rows, logits_parts, live_parts, new_files = [], [], [], [], []\n    sample_offset = 0\n    while step < total:\n        epoch, bindex = divmod(step, steps)\n        if not update_rows or bindex == 0:\n            order = source_order(len(data.fit), seed, epoch)\n            fresh = fresh_codes(data, seed, epoch)\n        positions = order[bindex * batch : (bindex + 1) * batch]\n        indices = data.fit[positions]\n        slots = np.full(len(indices), -1, dtype=np.int64)\n        codes = fresh[positions]\n        target = None\n        if bank is not None:\n            slots = (epoch + bank[\"slot_offsets\"][positions]) % bank[\"counts\"][positions]\n            codes = bank[\"codes\"][positions, slots]\n            target = torch.from_numpy(np.array(bank[\"logits\"][bank[\"offsets\"][positions] + slots])).to(\n                device\n            )\n        features, y = data.inputs(indices, codes, device)\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        rate = lr_for(step, total, warm)\n        optimizer.param_groups[0][\"lr\"] = rate\n        if device.type == \"cuda\":\n            torch.cuda.synchronize(device)\n            torch.cuda.reset_peak_memory_stats(device)\n        begin = time.perf_counter()\n        if arm == \"live_kd\":\n            with torch.no_grad():\n                target = teacher(features)\n            live_parts.append(target.cpu().numpy())\n        preds = model(features)\n        ce = F.cross_entropy(preds, y, reduction=\"none\")\n        kl = None\n        if target is None:\n            obj = ce.mean()\n        else:\n            kl = F.kl_div((preds / 4).log_softmax(-1), (target / 4).softmax(-1), reduction=\"none\").sum(1)\n            obj = (0.5 * ce + 8 * kl).mean()\n        if not torch.isfinite(obj):\n            raise FloatingPointError(f\"Nonfinite objective at update {step}\")\n        obj.backward()\n        norm = torch.linalg.vector_norm(\n            torch.stack([p.grad.detach().norm() for p in model.parameters() if p.grad is not None])\n        )\n        if not torch.isfinite(norm):\n            raise FloatingPointError(f\"Nonfinite gradient at update {step}\")\n        optimizer.step()\n        if device.type == \"cuda\":\n            torch.cuda.synchronize(device)\n        seconds = time.perf_counter() - begin\n        logits = preds.detach().cpu().numpy()\n        probabilities = preds.detach().softmax(1).gather(1, y[:, None]).ravel().cpu().numpy()\n        prediction = logits.argmax(1)\n        labels = data.labels[indices]\n        count = len(indices)\n        correct = int(np.sum(prediction == labels))\n        ces, kls, objs = (\n            ce.detach().sum().item(),\n            (kl.detach().sum().item() if kl is not None else 0.0),\n            obj.item() * count,\n        )\n        partial = [\n            partial[0] + count,\n            partial[1] + ces,\n            partial[2] + kls,\n            partial[3] + objs,\n            partial[4] + correct,\n        ]\n        update_rows.append(\n            (\n                step + 1,\n                epoch + 1,\n                bindex,\n                count,\n                ces,\n                kls if kl is not None else \"\",\n                objs,\n                correct,\n                rate,\n                norm.item(),\n                seconds,\n                torch.cuda.max_memory_allocated(device) if device.type == \"cuda\" else \"\",\n            )\n        )\n        exposure.extend(\n            (\n                step + 1,\n                step,\n                epoch + 1,\n                sample_offset + j,\n                data.ids[i],\n                int(labels[j]),\n                int(codes[j]),\n                int(slots[j]) if bank is not None else \"\",\n                int(prediction[j]),\n                float(probabilities[j]),\n            )\n            for j, i in enumerate(indices)\n        )\n        logits_parts.append(logits)\n        sample_offset += count\n        step += 1\n        boundary = step % steps == 0\n        if boundary:\n            val, val_correct, nll = evaluate(model, data, data.validation, device, batch)\n            folder = f\"epochs/{epoch+1:03d}\"\n            weights = folder + \"/weights.pt\"\n            torch_write(store.path(weights), model.state_dict())\n            np_write(store.path(folder + \"/validation_logits.npy\"), val)\n            csv_write(\n                store.path(folder + \"/validation.csv\"),\n                [\"row\", \"image_id\", \"label\", \"prediction\"],\n                ((j, data.ids[i], data.labels[i], val[j].argmax()) for j, i in enumerate(data.validation)),\n            )\n            csv_write(\n                store.path(folder + \"/metrics.csv\"),\n                [\n                    \"epoch\",\n                    \"fit_n\",\n                    \"ce_sum\",\n                    \"kl_sum\",\n                    \"objective_sum\",\n                    \"fit_correct\",\n                    \"validation_n\",\n                    \"validation_correct\",\n                    \"validation_nll\",\n                ],\n                [\n                    (\n                        epoch + 1,\n                        *partial[:2],\n                        partial[2] if target is not None else \"\",\n                        *partial[3:],\n                        len(data.validation),\n                        val_correct,\n                        nll,\n                    )\n                ],\n            )\n            new_files.extend(\n                [\n                    weights,\n                    folder + \"/validation_logits.npy\",\n                    folder + \"/validation.csv\",\n                    folder + \"/metrics.csv\",\n                ]\n            )\n            if best is None or (val_correct, -nll) > (best[\"correct\"], -best[\"nll\"]):\n                best = dict(epoch=epoch + 1, correct=val_correct, nll=nll, weights=weights)\n            partial = [0, 0.0, 0.0, 0.0, 0]\n        should_stop = (until is not None and step >= until) or stop_requested()\n        if boundary or len(update_rows) >= config[\"commit_updates\"] or should_stop:\n            folder = f\"updates/{step:08d}\"\n            csv_write(\n                store.path(folder + \"/updates.csv\"),\n                [\n                    \"update\",\n                    \"epoch\",\n                    \"batch\",\n                    \"n\",\n                    \"ce_sum\",\n                    \"kl_sum\",\n                    \"objective_sum\",\n                    \"correct\",\n                    \"learning_rate\",\n                    \"gradient_l2\",\n                    \"seconds\",\n                    \"peak_allocated_bytes\",\n                ],\n                update_rows,\n            )\n            csv_write(\n                store.path(folder + \"/exposure.csv\"),\n                [\n                    \"update\",\n                    \"weight_version\",\n                    \"epoch\",\n                    \"logit_row\",\n                    \"image_id\",\n                    \"label\",\n                    \"crop_flip_code\",\n                    \"bank_slot\",\n                    \"prediction\",\n                    \"true_probability\",\n                ],\n                exposure,\n            )\n            np_write(store.path(folder + \"/student_logits.npy\"), np.concatenate(logits_parts))\n            new_files.extend(\n                folder + \"/\" + n for n in (\"updates.csv\", \"exposure.csv\", \"student_logits.npy\")\n            )\n            if live_parts:\n                name = folder + \"/live_teacher_logits.npy\"\n                np_write(store.path(name), np.concatenate(live_parts))\n                new_files.append(name)\n            state = dict(\n                update=step,\n                complete=step == total,\n                model=model.state_dict(),\n                optimizer=optimizer.state_dict(),\n                rng=rng_state(),\n                best=best,\n                partial=partial,\n                scheduler_update=step,\n                scaler=None,\n            )\n            if state[\"complete\"]:\n                json_write(\n                    store.path(\"ENDPOINT.json\"),\n                    dict(\n                        final=f\"epochs/{epochs:03d}/weights.pt\",\n                        best=best,\n                        update=step,\n                        fit_images=len(data.fit),\n                        validation_images=len(data.validation),\n                    ),\n                )\n                new_files.append(\"ENDPOINT.json\")\n            store.commit(state, new_files, fault=\"before_pointer\" if step == fault_update else None)\n            on_commit(store, state[\"complete\"] or should_stop)\n            exposure, update_rows, logits_parts, live_parts, new_files = [], [], [], [], []\n            sample_offset = 0\n            if boundary:\n                print(\n                    f\"{config['model']} {arm} epoch {epoch+1}/{epochs}, durable update {step}\", flush=True\n                )\n            if should_stop:\n                return state\n    return state\n", "kd_fit_io.py": "\"\"\"Durable local tensors, direct evidence files and a complete-update checkpoint.\"\"\"\n\nimport csv\nimport hashlib\nimport json\nimport os\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\n\ndef canonical(value):\n    return json.dumps(value, sort_keys=True, separators=(\",\", \":\"), allow_nan=False).encode()\n\n\ndef hash_object(value):\n    return hashlib.sha256(canonical(value)).hexdigest()\n\n\ndef file_info(path):\n    h = hashlib.sha256()\n    with Path(path).open(\"rb\") as reader:\n        for chunk in iter(lambda: reader.read(1024 * 1024), b\"\"):\n            h.update(chunk)\n    return {\"bytes\": Path(path).stat().st_size, \"sha256\": h.hexdigest()}\n\n\ndef safe_path(root, relative):\n    if Path(relative).is_absolute() or \"..\" in Path(relative).parts:\n        raise ValueError(\"Invalid relative artifact path\")\n    root = Path(root).resolve()\n    p = (root / relative).resolve()\n    if not p.is_relative_to(root) or p == root:\n        raise ValueError(\"Artifact path outside its run directory\")\n    return p\n\n\ndef write_atomic(path, writer, binary=True):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    part = path.with_name(path.name + \".partial\")\n    mode = \"wb\" if binary else \"w\"\n    options = {} if binary else {\"encoding\": \"utf-8\", \"newline\": \"\"}\n    with part.open(mode, **options) as handle:\n        writer(handle)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(part, path)\n    if os.name == \"posix\":\n        directory = os.open(path.parent, os.O_RDONLY)\n        try:\n            os.fsync(directory)\n        finally:\n            os.close(directory)\n\n\ndef json_write(path, value):\n    write_atomic(path, lambda f: f.write(canonical(value)))\n\n\ndef json_read(path):\n    return json.loads(Path(path).read_bytes())\n\n\ndef csv_write(path, fields, rows):\n    def write(f):\n        writer = csv.writer(f, lineterminator=\"\\n\")\n        writer.writerow(fields)\n        writer.writerows(rows)\n\n    write_atomic(path, write, binary=False)\n\n\ndef np_write(path, array):\n    write_atomic(path, lambda f: np.save(f, array, allow_pickle=False))\n\n\ndef torch_write(path, obj):\n    write_atomic(path, lambda f: torch.save(obj, f))\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.use_deterministic_algorithms(True)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.backends.cuda.matmul.allow_tf32 = False\n\n\ndef rng_state():\n    return {\n        \"python\": random.getstate(),\n        \"numpy\": np.random.get_state(),\n        \"torch\": torch.get_rng_state(),\n        \"cuda\": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n    }\n\n\ndef rng_restore(state):\n    random.setstate(state[\"python\"])\n    np.random.set_state(state[\"numpy\"])\n    torch.set_rng_state(state[\"torch\"])\n    if state[\"cuda\"]:\n        if len(state[\"cuda\"]) != torch.cuda.device_count():\n            raise RuntimeError(\"Checkpoint GPU topology differs\")\n        torch.cuda.set_rng_state_all(state[\"cuda\"])\n\n\nclass Checkpoints:\n    \"\"\"Only POINTER.json selects committed evidence. Unselected crash tails are ignored.\"\"\"\n\n    def __init__(self, root, identity):\n        self.root = Path(root)\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.identity = identity\n        self.tag = hash_object(identity)\n        self.pointer_path = self.root / \"POINTER.json\"\n        self.pointer = json_read(self.pointer_path) if self.pointer_path.exists() else None\n        if self.pointer and self.pointer[\"identity\"] != self.tag:\n            raise RuntimeError(\"Checkpoint identity mismatch. Preserve the existing run.\")\n\n    def path(self, relative):\n        return safe_path(self.root, relative)\n\n    def load(self):\n        if not self.pointer:\n            return None\n        for name, meta in self.pointer[\"files\"].items():\n            if file_info(self.path(name)) != meta:\n                raise RuntimeError(\"Corrupt committed file: \" + name)\n        state = torch.load(self.path(self.pointer[\"state_file\"]), map_location=\"cpu\", weights_only=False)\n        if state[\"identity\"] != self.tag or state[\"generation\"] != self.pointer[\"generation\"]:\n            raise RuntimeError(\"State and selecting pointer disagree\")\n        return state[\"payload\"]\n\n    def commit(self, payload, names, fault=None):\n        previous = self.pointer or {}\n        generation = previous.get(\"generation\", 0) + 1\n        files = dict(previous.get(\"files\", {}))\n        if previous:\n            files.pop(previous[\"state_file\"])\n        for name in names:\n            info = file_info(self.path(name))\n            if name in files and files[name] != info:\n                raise RuntimeError(\"Committed evidence would change: \" + name)\n            files[name] = info\n        state_name = f\"rolling/state_{generation % 2}.pt\"\n        torch_write(self.path(state_name), dict(identity=self.tag, generation=generation, payload=payload))\n        files[state_name] = file_info(self.path(state_name))\n        if fault == \"before_pointer\":\n            raise InterruptedError(\"Injected power loss before pointer commit\")\n        row = {\n            \"generation\": generation,\n            \"identity\": self.tag,\n            \"update\": payload.get(\"update\", 0),\n            \"previous_chain\": previous.get(\"chain\"),\n            \"state_sha256\": files[state_name][\"sha256\"],\n            \"new_files\": {name: files[name] for name in names},\n        }\n        chain = hash_object(row)\n        ledger_name = f\"ledger/{generation:06d}.json\"\n        json_write(self.path(ledger_name), row)\n        files[ledger_name] = file_info(self.path(ledger_name))\n        pointer = dict(\n            identity=self.tag,\n            generation=generation,\n            chain=chain,\n            state_file=state_name,\n            update=payload.get(\"update\", 0),\n            complete=payload.get(\"complete\", False),\n            files=files,\n        )\n        json_write(self.pointer_path, pointer)\n        self.pointer = pointer\n        return pointer\n\n    def snapshot_files(self):\n        if self.pointer is None:\n            return {}\n        return {**self.pointer[\"files\"], \"POINTER.json\": file_info(self.pointer_path)}\n", "kd_hub.py": "\"\"\"One-owner, revision-pinned publication of direct CSV and native training evidence.\"\"\"\n\nimport hashlib\nimport random\nimport re\nimport time\nimport uuid\nfrom pathlib import Path\n\nfrom kd_fit_io import canonical, file_info, json_read, safe_path, write_atomic\n\n\ndef retry_delay(exc, attempt):\n    response = getattr(exc, \"response\", None)\n    status = getattr(response, \"status_code\", None)\n    if status not in (429, 500, 502, 503, 504) and (\n        status is not None\n        or not any(s in type(exc).__name__.lower() for s in (\"timeout\", \"connection\", \"transport\"))\n    ):\n        return None\n    headers = getattr(response, \"headers\", {}) or {}\n    value = headers.get(\"Retry-After\", \"0\")\n    try:\n        wait = float(value)\n    except ValueError:\n        from email.utils import parsedate_to_datetime\n\n        wait = max(0, parsedate_to_datetime(value).timestamp() - time.time())\n    reset = re.search(r\"\\bt=(\\d+)\", headers.get(\"RateLimit\", \"\"))\n    return (\n        max(wait, float(reset.group(1)) if reset else 0, min(45, 2**attempt))\n        + random.SystemRandom().random()\n    )\n\n\nclass Hub:\n    def __init__(self, repo, token, log, namespace=\"kd-view-budget/v1\"):\n        from huggingface_hub import HfApi\n        import requests\n\n        if not token or not re.fullmatch(r\"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+\", repo):\n            raise ValueError(\"A dedicated existing dataset repository and HF_TOKEN are required\")\n        self.repo, self.namespace, self.log = repo, namespace, Path(log)\n        self.api = HfApi(token=token)\n        self.http = requests.Session()\n        self.http.headers.update(Authorization=\"Bearer \" + token)\n        self.owner = uuid.uuid4().hex\n        self.last_verified = {}\n\n    def record(self, op, **fields):\n        self.log.parent.mkdir(parents=True, exist_ok=True)\n        with self.log.open(\"ab\") as f:\n            f.write(canonical(dict(operation=op, unix_time=time.time(), **fields)) + b\"\\n\")\n\n    def call(self, fn):\n        for attempt in range(6):\n            try:\n                return fn()\n            except Exception as exc:\n                delay = retry_delay(exc, attempt)\n                self.record(\"request_failure\", error_type=type(exc).__name__, delay=delay)\n                if delay is None or attempt == 5:\n                    raise\n                deadline = time.monotonic() + delay\n                while time.monotonic() < deadline:\n                    time.sleep(min(30, max(0, deadline - time.monotonic())))\n\n    def head(self):\n        return self.call(lambda: self.api.repo_info(self.repo, repo_type=\"dataset\").sha)\n\n    def read(self, path, rev):\n        def fetch():\n            with self.http.get(\n                f\"https://huggingface.co/datasets/{self.repo}/resolve/{rev}/{self.namespace}/{path}\",\n                timeout=(10, 90),\n            ) as response:\n                if response.status_code == 404:\n                    return None\n                response.raise_for_status()\n                return response.json()\n\n        return self.call(fetch)\n\n    def write_commit(self, changes, label, claim=False, release=False):\n        from huggingface_hub import CommitOperationAdd\n\n        for attempt in range(6):\n            head = self.head()\n            lease = self.read(\"OWNER.json\", head) or {}\n            other = lease.get(\"owner\") not in (None, self.owner)\n            if (claim and other and lease.get(\"expires\", 0) > time.time()) or (\n                not claim and lease.get(\"owner\") != self.owner\n            ):\n                raise RuntimeError(\"Another publisher owns the KD namespace\")\n            lease = {\n                \"owner\": None if release else self.owner,\n                \"expires\": 0 if release else time.time() + 2700,\n            }\n            values = {**changes, \"OWNER.json\": canonical(lease)}\n            ops = [\n                CommitOperationAdd(path_in_repo=self.namespace + \"/\" + n, path_or_fileobj=v)\n                for n, v in values.items()\n            ]\n            try:\n                rev = self.api.create_commit(\n                    self.repo,\n                    repo_type=\"dataset\",\n                    parent_commit=head,\n                    operations=ops,\n                    commit_message=\"KD view study \" + label,\n                ).oid\n                self.record(\"commit\", revision=rev, files=len(ops), label=label)\n                return rev\n            except Exception as exc:\n                code = getattr(getattr(exc, \"response\", None), \"status_code\", None)\n                if code in (409, 412) and attempt < 5:\n                    continue\n                # A timeout may follow successful selection or release. Recognize its marker.\n                current = self.head()\n                if self.read(\"OWNER.json\", current) == lease and all(\n                    self.read(n, current) == __import__(\"json\").loads(v)\n                    for n, v in changes.items()\n                    if isinstance(v, bytes) and n.endswith(\".json\")\n                ):\n                    return current\n                wait = retry_delay(exc, attempt)\n                if wait is None or attempt == 5:\n                    raise\n                time.sleep(wait)\n        raise RuntimeError(\"Could not acquire a consistent Hub parent revision\")\n\n    def claim(self):\n        self.write_commit({}, \"claim writer\", claim=True)\n\n    def release(self):\n        self.write_commit({}, \"release writer\", release=True)\n\n    def selected(self, unit):\n        revision = self.head()\n        return self.read(unit + \"/MANIFEST.json\", revision), revision\n\n    def publish(self, unit, store, extras=None):\n        local = {**store.snapshot_files(), **(extras or {})}\n        if not local:\n            return None\n        old, _ = self.selected(unit)\n        if old and old[\"identity\"] != store.tag:\n            raise RuntimeError(\"Remote scientific identity differs\")\n        entries = {}\n        for name, info in local.items():\n            path = Path(name)\n            remote = (path.parent / (path.stem + \"-\" + info[\"sha256\"][:24] + path.suffix)).as_posix()\n            entries[name] = {**info, \"object\": unit + \"/objects/\" + remote}\n        changed = [n for n, v in entries.items() if not old or old[\"files\"].get(n) != v]\n        for start in range(0, len(changed), 48):\n            names = changed[start : start + 48]\n            changes = {}\n            for name in names:\n                source = store.path(name)\n                if file_info(source) != local[name]:\n                    raise RuntimeError(\"File changed during snapshot: \" + name)\n                changes[entries[name][\"object\"]] = str(source)\n            self.write_commit(changes, \"stage evidence\")\n        revision = self.head()\n        for start in range(0, len(changed), 48):\n            names = changed[start : start + 48]\n            paths = [self.namespace + \"/\" + entries[n][\"object\"] for n in names]\n            infos = self.call(\n                lambda: self.api.get_paths_info(self.repo, paths, repo_type=\"dataset\", revision=revision)\n            )\n            by_name = {row.path: row for row in infos}\n            for name, path in zip(names, paths):\n                info = by_name[path]\n                if info.size != local[name][\"bytes\"]:\n                    raise RuntimeError(\"Remote byte count mismatch: \" + name)\n                lfs = getattr(info, \"lfs\", None)\n                if lfs:\n                    actual = lfs[\"sha256\"] if isinstance(lfs, dict) else lfs.sha256\n                    if actual != local[name][\"sha256\"]:\n                        raise RuntimeError(\"Remote LFS digest mismatch\")\n                else:\n                    body = store.path(name).read_bytes()\n                    git_id = hashlib.sha1(b\"blob \" + str(len(body)).encode() + b\"\\0\" + body).hexdigest()\n                    if info.blob_id != git_id:\n                        raise RuntimeError(\"Remote Git blob mismatch\")\n        snapshot = {\n            \"identity\": store.tag,\n            \"generation\": store.pointer[\"generation\"],\n            \"chain\": store.pointer[\"chain\"],\n            \"complete\": store.pointer[\"complete\"],\n            \"files\": entries,\n        }\n        rev = self.write_commit({unit + \"/MANIFEST.json\": canonical(snapshot)}, \"select \" + unit)\n        if self.read(unit + \"/MANIFEST.json\", rev) != snapshot:\n            raise RuntimeError(\"Selecting manifest failed verification\")\n        self.last_verified[unit] = rev\n        self.record(\"verified_manifest\", unit=unit, revision=rev, generation=snapshot[\"generation\"])\n        return rev\n\n    def fetch(self, unit, manifest, rev, folder, names=None):\n        for name in (manifest[\"files\"] if names is None else names):\n            meta = manifest[\"files\"][name]\n            target = safe_path(folder, name)\n            if target.exists() and file_info(target) == {k: meta[k] for k in (\"bytes\", \"sha256\")}:\n                continue\n            obj = meta[\"object\"]\n            if not obj.startswith(unit + \"/objects/\") or \"..\" in Path(obj).parts:\n                raise RuntimeError(\"Invalid remote object name\")\n\n            def transfer():\n                url = f\"https://huggingface.co/datasets/{self.repo}/resolve/{rev}/{self.namespace}/{obj}\"\n                with self.http.get(url, timeout=(15, 120), stream=True) as response:\n                    response.raise_for_status()\n\n                    def write(handle):\n                        h = hashlib.sha256()\n                        length = 0\n                        for chunk in response.iter_content(1024 * 1024):\n                            handle.write(chunk)\n                            h.update(chunk)\n                            length += len(chunk)\n                        if length != meta[\"bytes\"] or h.hexdigest() != meta[\"sha256\"]:\n                            raise RuntimeError(\"Pinned download failed its digest\")\n\n                    write_atomic(target, write)\n\n            self.call(transfer)\n\n    def reconcile(self, unit, store):\n        remote, rev = self.selected(unit)\n        if not remote:\n            return\n        if remote[\"identity\"] != store.tag:\n            raise RuntimeError(\"Remote identity differs from requested fit\")\n        local = store.pointer\n        if local and local[\"generation\"] >= remote[\"generation\"]:\n            name = f\"ledger/{remote['generation']:06d}.json\"\n            if local[\"files\"].get(name, {}).get(\"sha256\") != remote[\"files\"][name][\"sha256\"]:\n                raise RuntimeError(\"Local and remote checkpoint histories diverge\")\n            return\n        if local:\n            name = f\"ledger/{local['generation']:06d}.json\"\n            if remote[\"files\"].get(name, {}).get(\"sha256\") != local[\"files\"][name][\"sha256\"]:\n                raise RuntimeError(\"Remote checkpoint has a different history\")\n        names = [n for n in remote[\"files\"] if n != \"POINTER.json\"] + [\"POINTER.json\"]\n        self.fetch(unit, remote, rev, store.root, names)\n        store.pointer = json_read(store.pointer_path)\n        store.load()\n        self.record(\"restored_verified_bytes\", unit=unit, revision=rev, generation=remote[\"generation\"])\n", "kd_meter.py": "\"\"\"Per-board readings over synchronized windows. Missing counters stay missing.\"\"\"\n\nimport math\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\nimport torch\n\nfrom kd_fit_io import csv_write\n\n\ndef energy_from_rows(rows, duration):\n    if not rows or duration <= 0:\n        return None, \"unavailable\", 0.0\n    t0, t1 = rows[0][0], rows[-1][0]\n    covered = max(0, t1 - t0) / duration\n    counters = [r[2] for r in rows if r[2] is not None]\n    if (\n        covered >= 0.98\n        and duration - (t1 - t0) <= 0.5\n        and rows[0][2] is not None\n        and rows[-1][2] is not None\n        and rows[-1][2] > rows[0][2]\n        and all(a <= b for a, b in zip(counters, counters[1:]))\n    ):\n        return (rows[-1][2] - rows[0][2]) / 1000, \"nvml_counter_mJ\", covered\n    valid = [r for r in rows if r[1] is not None]\n    if len(valid) < 2:\n        return None, \"no_power_samples\", 0.0\n    coverage = (valid[-1][0] - valid[0][0]) / duration\n    intervals = [b[0] - a[0] for a, b in zip(valid, valid[1:])]\n    if coverage < 0.98 or max(intervals) > 1 or valid[0][0] - t0 > 0.25 or t1 - valid[-1][0] > 0.25:\n        return None, \"insufficient_coverage\", coverage\n    joules = math.fsum((a[1] + b[1]) * 0.5 * (b[0] - a[0]) for a, b in zip(valid, valid[1:]))\n    return joules, \"trapezoidal_power_W\", coverage\n\n\nclass BoardWindow:\n    def __init__(self, folder, phase, device):\n        self.folder, self.phase, self.device = Path(folder), phase, device\n        self.folder.mkdir(parents=True, exist_ok=True)\n        self.nv = None\n        self.handles = []\n        self.error = \"\"\n        self.active_uuid = \"\"\n        if device.type == \"cuda\":\n            properties = torch.cuda.get_device_properties(device)\n            self.active_uuid = str(getattr(properties, \"uuid\", \"\")).lower().removeprefix(\"gpu-\")\n        try:\n            import pynvml as nv\n\n            nv.nvmlInit()\n            self.nv = nv\n            for i in range(nv.nvmlDeviceGetCount()):\n                handle = nv.nvmlDeviceGetHandleByIndex(i)\n                self.handles.append((str(nv.nvmlDeviceGetUUID(handle)), handle))\n        except Exception as exc:\n            self.error = type(exc).__name__\n        self.rows = [[] for _ in self.handles]\n        self.stopped = threading.Event()\n\n    def sample(self):\n        for i, (_, handle) in enumerate(self.handles):\n            now = time.monotonic()\n            try:\n                watts = self.nv.nvmlDeviceGetPowerUsage(handle) / 1000\n            except Exception:\n                watts = None\n            try:\n                energy = self.nv.nvmlDeviceGetTotalEnergyConsumption(handle)\n            except Exception:\n                energy = None\n            self.rows[i].append((now, watts, energy))\n\n    def start(self):\n        if self.device.type == \"cuda\":\n            torch.cuda.synchronize(self.device)\n        self.begin = time.monotonic()\n        self.sample()\n\n        def monitor():\n            while not self.stopped.wait(0.2):\n                self.sample()\n\n        self.thread = threading.Thread(target=monitor, daemon=True)\n        self.thread.start()\n        return self\n\n    def finish(self):\n        if self.device.type == \"cuda\":\n            torch.cuda.synchronize(self.device)\n        self.stopped.set()\n        self.thread.join()\n        self.sample()\n        elapsed = time.monotonic() - self.begin\n        key = uuid.uuid4().hex\n        raw = self.folder / (key + \"_samples.csv\")\n        summary = self.folder / (key + \"_phase.csv\")\n        csv_write(\n            raw,\n            [\"window\", \"phase\", \"nvml_index\", \"uuid\", \"monotonic_s\", \"power_W\", \"cumulative_mJ\"],\n            (\n                (key, self.phase, i, self.handles[i][0], t, p, e)\n                for i, rows in enumerate(self.rows)\n                for t, p, e in rows\n            ),\n        )\n        results = []\n        for i, rows in enumerate(self.rows):\n            joules, method, coverage = energy_from_rows(rows, elapsed)\n            unit_uuid = self.handles[i][0].lower().removeprefix(\"gpu-\")\n            active = int(unit_uuid == self.active_uuid) if self.active_uuid else \"\"\n            results.append(\n                (\n                    key,\n                    self.phase,\n                    i,\n                    self.handles[i][0],\n                    active,\n                    elapsed,\n                    joules,\n                    method,\n                    coverage,\n                    len(rows),\n                )\n            )\n        if not results:\n            results.append(\n                (key, self.phase, \"\", \"\", \"\", elapsed, \"\", self.error or \"no_NVML_devices\", 0, 0)\n            )\n        csv_write(\n            summary,\n            [\n                \"window\",\n                \"phase\",\n                \"nvml_index\",\n                \"uuid\",\n                \"active_board\",\n                \"wall_s\",\n                \"board_J\",\n                \"method\",\n                \"coverage\",\n                \"observations\",\n            ],\n            results,\n        )\n        if self.nv is not None:\n            try:\n                self.nv.nvmlShutdown()\n            except Exception:\n                pass\n        return raw, summary\n", "kd_qualify.py": "\"\"\"Disposable fits and independent fresh-process recovery checks for the primary runner.\"\"\"\n\nimport argparse\nimport csv\nimport json\nimport os\nimport subprocess\nimport sys\nimport tempfile\nfrom pathlib import Path\n\nos.environ.setdefault(\"CUBLAS_WORKSPACE_CONFIG\", \":4096:8\")\nimport numpy as np\nimport torch\n\nfrom cache_preflight import render_view\nfrom kd_fit_data import build_bank, toy_images\nfrom kd_fit_engine import model_for, train_fit\nfrom kd_fit_io import Checkpoints, file_info, json_read, json_write\n\n\ndef toy_config(arm=\"teacher\"):\n    return dict(seed=42, arm=arm, epochs=3, batch=8, warm_epochs=1, commit_updates=2, model=\"fixture\")\n\n\ndef unit(path, arm, device):\n    torch.set_num_threads(1)\n    data = toy_images()\n    cfg = toy_config(arm)\n    store = Checkpoints(path, {\"kind\": \"fixture\", \"cfg\": cfg, \"data\": data.identity, \"device\": str(device)})\n    return data, cfg, store\n\n\ndef resumed_worker(root, limit, fault, device):\n    data, cfg, store = unit(root, \"teacher\", device)\n    return train_fit(\n        store, data, cfg, torch.device(device), lambda *_: None, until=limit, fault_update=fault\n    )\n\n\ndef child(root, device, until=None, fault=None):\n    args = [\n        sys.executable,\n        \"-B\",\n        str(Path(__file__).resolve()),\n        \"--worker\",\n        str(root),\n        \"--device\",\n        str(device),\n    ]\n    if until is not None:\n        args += [\"--until\", str(until)]\n    if fault is not None:\n        args += [\"--fault\", str(fault)]\n    env = dict(os.environ)\n    env.pop(\"HF_TOKEN\", None)\n    result = subprocess.run(\n        args, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=240\n    )\n    if fault is None and result.returncode != 0:\n        raise RuntimeError(result.stdout[-4000:])\n    if fault is not None and (result.returncode == 0 or \"Injected power loss\" not in result.stdout):\n        raise RuntimeError(\"Expected pre-pointer fault was not observed: \" + result.stdout[-700:])\n\n\ndef same(left, right):\n    if torch.is_tensor(left):\n        return torch.equal(left, right)\n    if isinstance(left, np.ndarray):\n        return np.array_equal(left, right)\n    if isinstance(left, dict):\n        return left.keys() == right.keys() and all(same(v, right[k]) for k, v in left.items())\n    if isinstance(left, (list, tuple)):\n        return len(left) == len(right) and all(same(a, b) for a, b in zip(left, right))\n    return left == right\n\n\ndef predictions(store):\n    actual = {}\n    for name in sorted(store.pointer[\"files\"]):\n        if not name.endswith(\"/exposure.csv\"):\n            continue\n        logits = np.load(store.path(name).with_name(\"student_logits.npy\"), allow_pickle=False)\n        with store.path(name).open(encoding=\"utf-8\", newline=\"\") as f:\n            for row in csv.DictReader(f):\n                k = (int(row[\"update\"]), row[\"image_id\"])\n                if k in actual:\n                    raise RuntimeError(\"Repeated committed training observation\")\n                actual[k] = logits[int(row[\"logit_row\"])]\n    return actual\n\n\ndef verify_continuation(reference, restored, device):\n    _, _, a = unit(reference, \"teacher\", device)\n    _, _, b = unit(restored, \"teacher\", device)\n    s1, s2 = a.load(), b.load()\n    for field in (\"model\", \"optimizer\", \"rng\", \"partial\", \"update\", \"scheduler_update\", \"scaler\", \"best\"):\n        if not same(s1[field], s2[field]):\n            raise RuntimeError(\"Restart diverged on \" + field)\n    obs_a, obs_b = predictions(a), predictions(b)\n    if not same(obs_a, obs_b) or len(obs_a) != 72:\n        raise RuntimeError(\"Restart changed the committed augmented predictions\")\n    return dict(\n        updates=s1[\"update\"],\n        observations=len(obs_a),\n        exact_model_optimizer_rng=True,\n        exact_training_predictions=True,\n    )\n\n\ndef qualification(root, device=\"cpu\", all_arms=True):\n    device = torch.device(device)\n    root = Path(root)\n    root.mkdir(parents=True, exist_ok=True)\n    baseline, recovered = root / \"uninterrupted\", root / \"interrupted\"\n    child(baseline, device)\n    child(recovered, device, until=2)\n    child(recovered, device, fault=3)\n    if json_read(recovered / \"POINTER.json\")[\"update\"] != 2:\n        raise RuntimeError(\"Crash advanced the selecting pointer\")\n    child(recovered, device)\n    receipt = verify_continuation(baseline, recovered, device)\n    data, _, control = unit(baseline, \"teacher\", device)\n    teacher = model_for(\"fixture\", data.classes).to(device).eval()\n    endpoint = json_read(control.path(\"ENDPOINT.json\"))\n    teacher.load_state_dict(\n        torch.load(control.path(endpoint[\"best\"][\"weights\"]), map_location=device, weights_only=True)\n    )\n    xi = np.asarray([0, 3, 8])\n    vi = np.asarray([0, 5, 18])\n    actual, _ = data.inputs(xi, vi, device)\n    ref = torch.stack(\n        [render_view(torch.from_numpy(data.rgb[i].copy()), int(v), data.padding) for i, v in zip(xi, vi)]\n    )\n    expected = (\n        ref.float().to(device) / 255\n        - torch.tensor(data.mean, dtype=torch.float32, device=device)[None, :, None, None]\n    )\n    expected /= torch.tensor(data.std, dtype=torch.float32, device=device)[None, :, None, None]\n    if not torch.equal(actual, expected):\n        raise RuntimeError(\"Batched crop differs from independent crop replay\")\n    receipt[\"vectorized_crop_exact\"] = True\n    receipt[\"arms\"] = {}\n    if all_arms:\n        for arm in (\"hard_labels\", \"live_kd\", \"uniform\", \"shuffled\", \"entropy\", \"disagreement\"):\n            _, candidate, store = unit(root / \"arms\" / arm, arm, device)\n            bank = None\n            if arm not in (\"hard_labels\", \"live_kd\"):\n                cstore = Checkpoints(\n                    root / \"banks\" / arm, {\"fixture\": arm, \"seed\": 42, \"data\": data.identity}\n                )\n                bank = build_bank(cstore, data, teacher, device, 42, arm, lambda *_: None, group=16)\n                if cstore.pointer[\"update\"] != 96 or bank[\"logits\"].shape != (96, 4):\n                    raise RuntimeError(\"Incorrect cache call count or shape\")\n                # Actual selected teacher and stored target must agree on corresponding view tensors.\n                positions = np.arange(6)\n                slots = bank[\"counts\"][positions] - 1\n                samples, _ = data.inputs(data.fit[positions], bank[\"codes\"][positions, slots], device)\n                with torch.no_grad():\n                    error = float(\n                        np.max(\n                            np.abs(\n                                teacher(samples).cpu().numpy()\n                                - bank[\"logits\"][bank[\"offsets\"][positions] + slots]\n                            )\n                        )\n                    )\n                if error > 1e-5:\n                    raise RuntimeError(\"Fitted-teacher cache differs from live replay\")\n            state = train_fit(store, data, candidate, device, lambda *_: None, teacher=teacher, bank=bank)\n            observed = predictions(store)\n            if len(observed) != 72 or not state[\"complete\"]:\n                raise RuntimeError(\"Fixture fit did not complete its full exposure budget\")\n            from kd_review import audit_fit\n\n            audited = audit_fit(store, data, bank)\n            receipt[\"arms\"][arm] = dict(\n                updates=state[\"update\"],\n                observations=len(observed),\n                build_queries=96 if bank is not None else 0,\n                independently_reconstructed_loss_error=audited[\"max_per_image_loss_error\"],\n            )\n    # Corrupt a copy of a selected file and ensure verification fails.\n    from shutil import copytree\n\n    bad = root / \"corrupt-copy\"\n    copytree(recovered, bad)\n    _, _, broken = unit(bad, \"teacher\", device)\n    with broken.path(\"config.json\").open(\"ab\") as f:\n        f.write(b\"bad\")\n    try:\n        broken.load()\n    except RuntimeError as exc:\n        if \"Corrupt\" not in str(exc):\n            raise\n    else:\n        raise RuntimeError(\"Corruption was accepted\")\n    try:\n        Checkpoints(recovered, {\"incompatible\": True})\n    except RuntimeError:\n        receipt[\"incompatible_identity_rejected\"] = True\n    else:\n        raise RuntimeError(\"Incompatible identity accepted\")\n    receipt[\"corruption_rejected\"] = True\n    before = file_info(control.pointer_path)\n    child(baseline, device)\n    if before != file_info(control.pointer_path):\n        raise RuntimeError(\"Completed fit was rewritten\")\n    receipt[\"completed_run_skipped\"] = True\n    json_write(root / \"qualification.json\", receipt)\n    return receipt\n\n\nif __name__ == \"__main__\":\n    p = argparse.ArgumentParser()\n    p.add_argument(\"--worker\", type=Path)\n    p.add_argument(\"--device\", default=\"cpu\")\n    p.add_argument(\"--until\", type=int)\n    p.add_argument(\"--fault\", type=int)\n    p.add_argument(\"--qualify\", type=Path)\n    args = p.parse_args()\n    if args.worker:\n        resumed_worker(args.worker, args.until, args.fault, args.device)\n    else:\n        root = args.qualify or Path(tempfile.mkdtemp(prefix=\"kd-master-check-\"))\n        print(json.dumps(qualification(root, args.device), indent=2))\n", "kd_review.py": "\"\"\"Recompute fitted-run evidence and release final CIFAR-100 predictions after all five blocks.\"\"\"\n\nimport copy\nimport csv\nimport math\nimport tarfile\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nfrom cifar100_manifest import RECORD_BYTES\nfrom kd_fit_engine import evaluate, model_for\nfrom kd_fit_io import Checkpoints, csv_write, hash_object, json_read, json_write, np_write\n\n\ndef read_csv(path):\n    with Path(path).open(encoding=\"utf-8\", newline=\"\") as f:\n        return list(csv.DictReader(f))\n\n\ndef audit_fit(store, data, bank=None):\n    \"\"\"Verify committed augmented predictions, loss numerators, full source coverage, selection and denominator.\"\"\"\n    if not store.pointer or not store.pointer[\"complete\"]:\n        raise ValueError(\"A completed fit is required\")\n    cfg = json_read(store.path(\"config.json\"))\n    endpoint = json_read(store.path(\"ENDPOINT.json\"))\n    fit_ids = [data.ids[i] for i in data.fit]\n    positions = {image_id: i for i, image_id in enumerate(fit_ids)}\n    seen = np.zeros((cfg[\"epochs\"], len(fit_ids)), dtype=np.uint8)\n    total_obs = 0\n    total_updates = 0\n    largest = 0.0\n    for name in sorted(store.pointer[\"files\"]):\n        if not name.endswith(\"/updates.csv\"):\n            continue\n        folder = store.path(name).parent\n        up = read_csv(folder / \"updates.csv\")\n        exposure = read_csv(folder / \"exposure.csv\")\n        logits = np.load(folder / \"student_logits.npy\", allow_pickle=False)\n        if len(exposure) != len(logits):\n            raise ValueError(\"Unmatched training logit rows\")\n        target = None\n        if cfg[\"arm\"] == \"live_kd\":\n            target = np.load(folder / \"live_teacher_logits.npy\", allow_pickle=False)\n        if bank is not None:\n            cols = np.array([positions[r[\"image_id\"]] for r in exposure])\n            slots = np.array([int(r[\"bank_slot\"]) for r in exposure])\n            target = np.array(bank[\"logits\"][bank[\"offsets\"][cols] + slots])\n        cursor = 0\n        for u in up:\n            step = int(u[\"update\"])\n            n = int(u[\"n\"])\n            if step != total_updates + 1:\n                raise ValueError(\"Missing or repeated update\")\n            subset = exposure[cursor : cursor + n]\n            if len(subset) != n or any(\n                int(row[\"update\"]) != step\n                or int(row[\"weight_version\"]) != step - 1\n                or int(row[\"logit_row\"]) != cursor + j\n                for j, row in enumerate(subset)\n            ):\n                raise ValueError(\"Exposure cursor mismatch\")\n            ids = np.asarray([positions[row[\"image_id\"]] for row in subset])\n            labels = torch.from_numpy(data.labels[data.fit[ids]])\n            if labels.tolist() != [int(row[\"label\"]) for row in subset]:\n                raise ValueError(\"Training labels differ from manifest\")\n            epoch = int(u[\"epoch\"]) - 1\n            np.add.at(seen[epoch], ids, 1)\n            z = torch.from_numpy(logits[cursor : cursor + n]).double()\n            if int((z.argmax(1) == labels).sum()) != int(u[\"correct\"]):\n                raise ValueError(\"Training accuracy mismatch\")\n            ce = float(F.cross_entropy(z, labels, reduction=\"sum\"))\n            kl = 0.0\n            if target is not None:\n                t = torch.from_numpy(target[cursor : cursor + n]).double()\n                kl = float(F.kl_div((z / 4).log_softmax(1), (t / 4).softmax(1), reduction=\"sum\"))\n            elif cfg[\"arm\"] not in (\"hard_labels\", \"teacher\"):\n                raise ValueError(\"Cached-target loss audit requires its bank\")\n            objective = ce if target is None else 0.5 * ce + 8 * kl\n            discrepancies = [abs(ce - float(u[\"ce_sum\"])), abs(objective - float(u[\"objective_sum\"]))]\n            if target is not None:\n                discrepancies.append(abs(kl - float(u[\"kl_sum\"])))\n            error = max(discrepancies) / n\n            if error > 2e-5:\n                raise ValueError(f\"Per-example loss reconstruction exceeded tolerance at update {step}\")\n            largest = max(largest, error)\n            cursor += n\n            total_updates += 1\n        if cursor != len(exposure):\n            raise ValueError(\"Unconsumed training exposure\")\n        total_obs += cursor\n    if not np.all(seen == 1) or total_updates != endpoint[\"update\"]:\n        raise ValueError(\"Incomplete epoch exposure or update ledger\")\n    by_epoch = []\n    for e in range(1, cfg[\"epochs\"] + 1):\n        root = store.path(f\"epochs/{e:03d}\")\n        z = torch.from_numpy(np.load(root / \"validation_logits.npy\", allow_pickle=False)).double()\n        ref = read_csv(root / \"validation.csv\")\n        if [r[\"image_id\"] for r in ref] != [data.ids[i] for i in data.validation]:\n            raise ValueError(\"Validation cohort mismatch\")\n        y = torch.from_numpy(data.labels[data.validation])\n        correct = int((z.argmax(1) == y).sum())\n        nll = float(F.cross_entropy(z, y))\n        m = read_csv(root / \"metrics.csv\")[0]\n        if int(m[\"validation_correct\"]) != correct or abs(nll - float(m[\"validation_nll\"])) > 1e-7:\n            raise ValueError(\"Validation metric reconstruction mismatch\")\n        by_epoch.append((correct, -nll, -e))\n    winner = -max(by_epoch)[2]\n    if winner != endpoint[\"best\"][\"epoch\"]:\n        raise ValueError(\"Validation selection mismatch\")\n    return dict(\n        updates=total_updates,\n        observations=total_obs,\n        validation_per_epoch=len(data.validation),\n        selected_validation_epoch=winner,\n        final_epoch=cfg[\"epochs\"],\n        max_per_image_loss_error=largest,\n        exact_coverage_each_epoch=True,\n    )\n\n\ndef metrics(logits, labels, classes):\n    z = torch.from_numpy(logits).double()\n    y = torch.from_numpy(labels)\n    prob = z.softmax(1)\n    conf, pred = prob.max(1)\n    correct = pred == y\n    bins = torch.minimum((15 * conf).long(), torch.full_like(y, 14))\n    ece = 0.0\n    for b in range(15):\n        mask = bins == b\n        if mask.any():\n            ece += float(mask.double().mean() * abs(correct[mask].double().mean() - conf[mask].mean()))\n    confusion = np.bincount(labels * classes + pred.numpy(), minlength=classes**2).reshape(classes, classes)\n    supported = confusion.sum(1) > 0\n    macro = float(np.mean(confusion.diagonal()[supported] / confusion.sum(1)[supported]))\n    return int(correct.sum()), float(F.cross_entropy(z, y)), ece, macro, confusion\n\n\ndef board_totals(files):\n    windows = {}\n    for path in files:\n        for r in read_csv(path):\n            windows.setdefault(r[\"window\"], []).append(r)\n    active = reserved = wall = 0.0\n    active_ok = reserved_ok = bool(windows)\n    for rows in windows.values():\n        wall += float(rows[0][\"wall_s\"])\n        primary = [r for r in rows if r[\"active_board\"] == \"1\"]\n        if len(primary) == 1 and primary[0][\"board_J\"] != \"\":\n            active += float(primary[0][\"board_J\"])\n        else:\n            active_ok = False\n        if len(rows) == 2 and len({r[\"uuid\"] for r in rows}) == 2 and all(r[\"board_J\"] != \"\" for r in rows):\n            reserved += sum(float(r[\"board_J\"]) for r in rows)\n        else:\n            reserved_ok = False\n    return {\n        \"active_J\": active if active_ok else None,\n        \"both_reserved_J\": reserved if reserved_ok else None,\n        \"wall_s\": wall if windows else None,\n        \"windows\": len(windows),\n    }\n\n\ndef final_release(hub, base, archive, data, arms, seeds, source, protocol):\n    if len(seeds) != 5 or len(arms) != 6:\n        raise ValueError(\"Expected all five predeclared primary blocks and six arms\")\n    selection = {}\n    for s in seeds:\n        for arm in arms:\n            run = f\"v1.cifar100.resnet18.{arm}.{s}\"\n            manifest, rev = hub.selected(run)\n            audit, audit_rev = hub.selected(run + \"-review\")\n            if not manifest or not manifest[\"complete\"] or not audit or not audit[\"complete\"]:\n                raise RuntimeError(\"Final labels remain sealed until this fit and review exist: \" + run)\n            expected_audit = hash_object(\n                {\"fit\": manifest[\"identity\"], \"chain\": manifest[\"chain\"], \"review_source\": source}\n            )\n            if audit[\"identity\"] != expected_audit:\n                raise RuntimeError(\"The evidence audit does not identify this exact completed fit\")\n            with tempfile.TemporaryDirectory(dir=base) as temp:\n                hub.fetch(run, manifest, rev, temp, [\"config.json\", \"ENDPOINT.json\"])\n                cfg = json_read(Path(temp) / \"config.json\")\n                ep = json_read(Path(temp) / \"ENDPOINT.json\")\n                if (\n                    cfg[\"contract\"][\"data\"] != data.identity\n                    or cfg[\"contract\"][\"source\"] != source\n                    or cfg[\"contract\"][\"protocol_sha256\"] != protocol\n                    or (cfg[\"seed\"], cfg[\"arm\"], cfg[\"model\"]) != (s, arm, \"resnet18\")\n                    or cfg[\"epochs\"] != 60\n                    or ep[\"final\"] != \"epochs/060/weights.pt\"\n                ):\n                    raise RuntimeError(\"Model does not match the frozen primary campaign\")\n            selection[run] = (manifest, rev, ep[\"final\"], audit_rev)\n    name = \"primary-final-review\"\n    # Bind stable selected file revisions by content. Unrelated later repository commits do not change identity.\n    identity = {\n        \"data\": data.identity,\n        \"source\": source,\n        \"protocol\": protocol,\n        \"endpoints\": {n: m[\"files\"][p][\"sha256\"] for n, (m, r, p, a) in selection.items()},\n    }\n    store = Checkpoints(base / \"final_review\", identity)\n    hub.reconcile(name, store)\n    if store.pointer and store.pointer[\"complete\"]:\n        remote, _ = hub.selected(name)\n        if remote is None or remote[\"chain\"] != store.pointer[\"chain\"]:\n            store.load()\n            hub.publish(name, store)\n        return json_read(store.path(\"paired_result.json\"))\n    with tarfile.open(archive, \"r:gz\") as tar:\n        with tar.extractfile(\"cifar-100-binary/test.bin\") as stream:\n            raw = np.frombuffer(stream.read(), np.uint8).reshape(10000, RECORD_BYTES)\n    testing = copy.copy(data)\n    testing.rgb = raw[:, 2:].reshape(-1, 3, 32, 32)\n    testing.labels = raw[:, 1].astype(np.int64)\n    testing.ids = [f\"test/{i:05d}\" for i in range(len(raw))]\n    manifest = read_csv(base / \"input\" / json_read(base / \"input\" / \"INPUT.json\")[\"manifest\"])\n    masked = np.array([r[\"role\"] == \"test_primary\" for r in manifest if r[\"image_id\"].startswith(\"test/\")])\n    if len(masked) != len(raw) or not masked.any():\n        raise ValueError(\"Final cohort mismatch\")\n    rows = []\n    files = []\n    counts = {}\n    for s in seeds:\n        for arm in arms:\n            run = f\"v1.cifar100.resnet18.{arm}.{s}\"\n            m, rev, weights, audit_rev = selection[run]\n            with tempfile.TemporaryDirectory(dir=base) as temp:\n                hub.fetch(run, m, rev, temp, [weights])\n                model = model_for(\"resnet18\", 100).cuda().eval()\n                model.load_state_dict(\n                    torch.load(Path(temp) / weights, map_location=\"cuda:0\", weights_only=True)\n                )\n                z, _, _ = evaluate(model, testing, np.arange(len(raw)), torch.device(\"cuda:0\"), 128)\n            pred_folder = f\"runs/{run}\"\n            np_write(store.path(pred_folder + \"/logits.npy\"), z)\n            csv_write(\n                store.path(pred_folder + \"/predictions.csv\"),\n                [\"row\", \"image_id\", \"label\", \"prediction\", \"eligible\"],\n                (\n                    (i, testing.ids[i], testing.labels[i], v, int(masked[i]))\n                    for i, v in enumerate(z.argmax(1))\n                ),\n            )\n            files.extend([pred_folder + \"/logits.npy\", pred_folder + \"/predictions.csv\"])\n            for label, mask in ((\"primary\", masked), (\"full\", np.ones(len(raw), dtype=bool))):\n                ok, nll, ece, macro, confusion = metrics(z[mask], testing.labels[mask], 100)\n                if label == \"primary\":\n                    counts[s, arm] = ok\n                rows.append(\n                    (\n                        s,\n                        arm,\n                        label,\n                        int(mask.sum()),\n                        ok,\n                        100 * ok / mask.sum(),\n                        nll,\n                        ece,\n                        macro,\n                        rev,\n                        audit_rev,\n                    )\n                )\n                path = pred_folder + f\"/{label}_confusion.csv\"\n                csv_write(\n                    store.path(path),\n                    [\"true_class\", *[f\"pred_{i}\" for i in range(100)]],\n                    ((i, *line) for i, line in enumerate(confusion)),\n                )\n                files.append(path)\n    csv_write(\n        store.path(\"results.csv\"),\n        [\n            \"seed\",\n            \"arm\",\n            \"cohort\",\n            \"n\",\n            \"correct\",\n            \"top1_pct\",\n            \"nll\",\n            \"ece\",\n            \"macro_recall\",\n            \"fit_revision\",\n            \"audit_revision\",\n        ],\n        rows,\n    )\n\n    def paired(control, method=\"disagreement\"):\n        deltas = np.array([100 * (counts[s, method] - counts[s, control]) / masked.sum() for s in seeds])\n        mean = float(deltas.mean())\n        radius = float(2.7764451052 * deltas.std(ddof=1) / math.sqrt(5))\n        return dict(deltas_pp=deltas.tolist(), mean_pp=mean, lower95=mean - radius, upper95=mean + radius)\n\n    primary, uniform = paired(\"shuffled\"), paired(\"uniform\")\n    result = dict(\n        denominator=int(masked.sum()),\n        seeds=list(seeds),\n        guided_minus_shuffled=primary,\n        guided_minus_uniform=uniform,\n        uniform_test_reached=primary[\"lower95\"] > 0,\n        practical_rule_passed=primary[\"lower95\"] > 0\n        and uniform[\"lower95\"] > 0\n        and primary[\"mean_pp\"] >= 0.3,\n    )\n    resource = {}\n    teacher_unit = \"v1.cifar100.resnet34.teacher.907\"\n    teacher_manifest, teacher_revision = hub.selected(teacher_unit)\n    if not teacher_manifest or not teacher_manifest[\"complete\"]:\n        raise RuntimeError(\"Missing completed teacher energy evidence\")\n    teacher_phases = [\n        n for n in teacher_manifest[\"files\"] if n.startswith(\"measurement/\") and n.endswith(\"_phase.csv\")\n    ]\n    with tempfile.TemporaryDirectory(dir=base) as temp:\n        hub.fetch(teacher_unit, teacher_manifest, teacher_revision, temp, teacher_phases)\n        resource[teacher_unit] = board_totals([Path(temp) / n for n in teacher_phases])\n    for s in seeds:\n        for arm in arms:\n            fit = f\"v1.cifar100.resnet18.{arm}.{s}\"\n            units = [fit]\n            if arm not in (\"hard_labels\", \"live_kd\"):\n                units.append(f\"v1.cifar100.cache.{arm}.{s}\")\n            for which in units:\n                if which in resource:\n                    continue\n                m, r = (selection[fit][0], selection[fit][1]) if which == fit else hub.selected(which)\n                if not m or not m[\"complete\"]:\n                    raise RuntimeError(\"Missing cache energy evidence\")\n                phase = [n for n in m[\"files\"] if n.startswith(\"measurement/\") and n.endswith(\"_phase.csv\")]\n                with tempfile.TemporaryDirectory(dir=base) as temp:\n                    hub.fetch(which, m, r, temp, phase)\n                    resource[which] = board_totals([Path(temp) / n for n in phase])\n    csv_write(\n        store.path(\"resources.csv\"),\n        [\"unit\", \"active_J\", \"both_reserved_J\", \"wall_s\", \"windows\"],\n        ((k, v[\"active_J\"], v[\"both_reserved_J\"], v[\"wall_s\"], v[\"windows\"]) for k, v in resource.items()),\n    )\n    reused = []\n    for method in (\"uniform\", \"shuffled\", \"entropy\", \"disagreement\"):\n        quality = paired(\"live_kd\", method)\n        for s in seeds:\n            b = resource[f\"v1.cifar100.cache.{method}.{s}\"][\"active_J\"]\n            l = resource[f\"v1.cifar100.resnet18.live_kd.{s}\"][\"active_J\"]\n            c = resource[f\"v1.cifar100.resnet18.{method}.{s}\"][\"active_J\"]\n            supported = all(v is not None for v in (b, l, c))\n            crossover = max(1, math.floor(b / (l - c)) + 1) if supported and l > c else None\n            status = (\n                \"finite\" if crossover else (\"no_finite_saving\" if supported else \"missing_board_reading\")\n            )\n            teacher_cost = resource[teacher_unit][\"active_J\"]\n            for r in (1, 3, 5):\n                reused.append(\n                    (\n                        s,\n                        method,\n                        r,\n                        b,\n                        c,\n                        l,\n                        (b + r * c) if supported else None,\n                        r * l if supported else None,\n                        teacher_cost,\n                        teacher_cost + b + r * c if supported and teacher_cost is not None else None,\n                        teacher_cost + r * l if supported and teacher_cost is not None else None,\n                        crossover,\n                        status,\n                        quality[\"lower95\"],\n                        quality[\"lower95\"] > -0.5,\n                    )\n                )\n    csv_write(\n        store.path(\"reuse_scenarios.csv\"),\n        [\n            \"seed\",\n            \"method\",\n            \"students_scenario\",\n            \"cache_build_J\",\n            \"cached_fit_J\",\n            \"live_fit_J\",\n            \"projected_cached_total_J\",\n            \"projected_live_total_J\",\n            \"teacher_training_J\",\n            \"greenfield_cached_J\",\n            \"greenfield_live_J\",\n            \"first_strict_saving_n\",\n            \"energy_status\",\n            \"paired_accuracy_lower95_pp\",\n            \"quality_margin_pass\",\n        ],\n        reused,\n    )\n    json_write(store.path(\"paired_result.json\"), result)\n    store.commit(\n        {\"update\": 30, \"complete\": True},\n        files + [\"results.csv\", \"paired_result.json\", \"resources.csv\", \"reuse_scenarios.csv\"],\n    )\n    hub.publish(name, store)\n    return result\n"}
EXPECTED = {'cache_preflight.py': 'bd4087187eceebdeaafd0b716fe82a4a685a2be082debe0be9e1c91c93e5b522', 'cifar100_manifest.py': '21e1f38d2bde1c847d5739b017410cf0e2ac387d94326a1c3e036873d2a2b144', 'kd_fit_io.py': 'df4e1d71c3d60294dbb21d390effe466dcd3e28adb3a1fb0f78578e21bdd14ca', 'kd_fit_data.py': '93bb3b1f5b4d6f7472d9f9bc7be7f023ff8c1e5e86a6a3fc49c2c31786ae3b7d', 'kd_fit_engine.py': 'd102854e6015ba5be09a68b6a93e2e78ae35bb2563525b672768a54a69af5c94', 'kd_meter.py': 'fe7a197bb893bb21302f229adfec8d8b3d756419683281aabd3c5ce2e9d38b47', 'kd_hub.py': 'a500222686147590215b09fb47d1b767a52c3c4879b2f0a42affe65519344d0b', 'kd_review.py': '471dbedcc217ace2dfa9ecb7e87076754d325f34ad6e2ce633d94fd66d9e455f', 'kd_qualify.py': '8e9744cd2d804ef02ca52e41c52a8c53712d3f9e27f3cb7b86cc06033137658c', 'kd_campaign.py': '793c5ed9a7afb6dcf12b2b021e135c4faf54935aa99275914d5c27d2311dc722', 'PROTOCOL.md': '698da1c50bfe7ff44c96790d68cba261918590a18c999e259eb3f64c002abe61'}
build = hashlib.sha256(json.dumps(EXPECTED,sort_keys=True).encode()).hexdigest()[:16]
runtime = Path(gettempdir()) / ('kd-view-runtime-' + build)
runtime.mkdir(parents=True,exist_ok=True)
for name, source in SOURCES.items():
    body=source.encode('utf-8')
    if hashlib.sha256(body).hexdigest()!=EXPECTED[name]: raise RuntimeError('Embedded source mismatch')
    target=runtime/name
    if target.exists() and target.read_bytes()!=body: raise RuntimeError('Runtime identity conflict')
    if not target.exists(): target.write_bytes(body)
sys.path.insert(0,str(runtime))
print('Verified embedded files',len(SOURCES),'build',build)

Verified embedded files 11 build af5e86e7fb1db85a


## Run the selected stage

For measured mode, attach the original CIFAR-100 binary archive, select two T4 GPUs and enable Internet and the Kaggle HF_TOKEN secret. The runner verifies remote continuation and local capacity before fitting. Preserve the executed notebook and its output receipts.

In [3]:
from kd_campaign import run_master
base = Path(os.environ.get('KD_MASTER_OUTPUT', '/kaggle/working/kd_view_primary' if Path('/kaggle/working').is_dir() else 'kd_view_primary'))
archive = Path(CIFAR100_ARCHIVE) if CIFAR100_ARCHIVE else None
if MODE != 'qualification' and archive is None:
    entries = sorted(Path('/kaggle/input').rglob('cifar-100-binary.tar.gz'))
    if len(entries) != 1: raise ValueError('Set CIFAR100_ARCHIVE to the original binary archive')
    archive = entries[0]
result = run_master(MODE,base,archive,HF_DATASET_REPO,SEED_BLOCKS,EXPECTED['PROTOCOL.md'])
print(json.dumps(result,indent=2))

fixture hard_labels epoch 1/3, durable update 3


fixture hard_labels epoch 2/3, durable update 6


fixture hard_labels epoch 3/3, durable update 9

fixture live_kd epoch 1/3, durable update 3


fixture live_kd epoch 2/3, durable update 6


fixture live_kd epoch 3/3, durable update 9


... (27 outputs left out)

fixture disagreement epoch 1/3, durable update 3


fixture disagreement epoch 2/3, durable update 6


fixture disagreement epoch 3/3, durable update 9

{
  "cpu": {
    "updates": 9,
    "observations": 72,
    "exact_model_optimizer_rng": true,
    "exact_training_predictions": true,
    "vectorized_crop_exact": true,
    "arms": {
      "hard_labels": {
        "updates": 9,
        "observations": 72,
        "build_queries": 0,
        "independently_reconstructed_loss_error": 1.1368875396655653e-07
      },
      "live_kd": {
        "updates": 9,
        "observations": 72,
        "build_queries": 0,
        "independently_reconstructed_loss_error": 4.308793933871513e-07
      },
      "uniform": {
        "updates": 9,
        "observations": 72,
        "build_queries": 96,
        "independently_reconstructed_loss_error": 3.500548046986296e-07
      },
      "shuffled": {
        "updates": 9,
        "observations": 72,
        "build_queries": 96,
        "independently_reconstructed_loss_error": 3.3756133510642883e-07
      },
      "entropy": {
        "updates": 9,
        "observations": 72,
        "build_queries": 96